<a href="https://colab.research.google.com/github/FaithChege254/FaithChege254/blob/main/Multilingual_Paza_Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 ============================================================
 MULTILINGUAL KENYAN ASR
 Paza Whisper Large-v3-Turbo Fine-Tuning
 ============================================================

 Project:
   Multilingual Speech-to-Text for Selected Kenyan Languages
 Languages:
   - Swahili
   - Kikuyu
   - Kalenjin

 Base Model:
   microsoft/paza-whisper-large-v3-turbo

 Training Platform:
   Google Colab

 Hardware:
   NVIDIA Tesla T4
Dataset:
 2,791 verified audio-transcription pairs


In [10]:
# ============================================================
# 01. PROJECT CONFIGURATION
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

MODEL_NAME = "microsoft/paza-whisper-large-v3-turbo"


# ------------------------------------------------------------
# Supported languages
# ------------------------------------------------------------

LANGUAGES = {
    "swahili": "sw",
    "kikuyu": "kik",
    "kalenjin": "kln",
}


# ------------------------------------------------------------
# Audio configuration
# ------------------------------------------------------------

SAMPLE_RATE = 16_000


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42


# ------------------------------------------------------------
# Project directories
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/multilingual_kenyan_asr")

DATASET_ROOT = PROJECT_ROOT / "dataset"

TRAIN_MANIFEST = DATASET_ROOT / "train.csv"
VALIDATION_MANIFEST = DATASET_ROOT / "validation.csv"

OUTPUT_ROOT = PROJECT_ROOT / "outputs"
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints"
FINAL_MODEL_ROOT = OUTPUT_ROOT / "final_model"


# ------------------------------------------------------------
# Create required directories
# ------------------------------------------------------------

for directory in [
    PROJECT_ROOT,
    DATASET_ROOT,
    OUTPUT_ROOT,
    CHECKPOINT_ROOT,
    FINAL_MODEL_ROOT,
]:
    directory.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Configuration summary
# ------------------------------------------------------------

print("=" * 60)
print("PROJECT CONFIGURATION")
print("=" * 60)

print(f"Model           : {MODEL_NAME}")
print(f"Languages       : {LANGUAGES}")
print(f"Sample rate     : {SAMPLE_RATE} Hz")
print(f"Random seed     : {SEED}")
print(f"Project root    : {PROJECT_ROOT}")
print(f"Dataset root    : {DATASET_ROOT}")
print(f"Output root     : {OUTPUT_ROOT}")

print("=" * 60)

PROJECT CONFIGURATION
Model           : microsoft/paza-whisper-large-v3-turbo
Languages       : {'swahili': 'sw', 'kikuyu': 'kik', 'kalenjin': 'kln'}
Sample rate     : 16000 Hz
Random seed     : 42
Project root    : /content/multilingual_kenyan_asr
Dataset root    : /content/multilingual_kenyan_asr/dataset
Output root     : /content/multilingual_kenyan_asr/outputs


In [11]:
# ============================================================
# 02. ENVIRONMENT VERIFICATION
# ============================================================

import sys
import platform
import torch


print("=" * 60)
print("ENVIRONMENT VERIFICATION")
print("=" * 60)


# ------------------------------------------------------------
# Python
# ------------------------------------------------------------

print(f"Python version        : {sys.version.split()[0]}")


# ------------------------------------------------------------
# Operating system
# ------------------------------------------------------------

print(f"Platform              : {platform.system()}")
print(f"Platform version      : {platform.release()}")


# ------------------------------------------------------------
# PyTorch
# ------------------------------------------------------------

print(f"PyTorch version       : {torch.__version__}")


# ------------------------------------------------------------
# CUDA
# ------------------------------------------------------------

print(f"CUDA available       : {torch.cuda.is_available()}")
print(f"PyTorch CUDA version : {torch.version.cuda}")


# ------------------------------------------------------------
# GPU
# ------------------------------------------------------------

if torch.cuda.is_available():

    gpu_name = torch.cuda.get_device_name(0)

    gpu_memory_gb = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )

    print(f"GPU                  : {gpu_name}")
    print(f"GPU memory           : {gpu_memory_gb:.2f} GB")

else:

    print("GPU                  : NOT AVAILABLE")


print("=" * 60)

ENVIRONMENT VERIFICATION
Python version        : 3.13.15
Platform              : Linux
Platform version      : 6.6.122+
PyTorch version       : 2.11.0+cu130
CUDA available       : True
PyTorch CUDA version : 13.0
GPU                  : Tesla T4
GPU memory           : 14.56 GB


In [12]:
# ============================================================
# 03. REPRODUCIBILITY
# ============================================================

import random
import numpy as np
import torch


def set_seed(seed: int = 42) -> None:
    """
    Set random seeds for reproducible experiments.
    """

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

print(f"[PASS] Random seed initialized to {SEED}")

[PASS] Random seed initialized to 42


In [13]:
# ============================================================
# 04. LOGGING
# ============================================================

import logging


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)


logger = logging.getLogger("multilingual_kenyan_asr")

logger.info("Project environment initialized.")
logger.info(f"Model: {MODEL_NAME}")
logger.info(f"Languages: {', '.join(LANGUAGES.keys())}")

In [14]:
# ============================================================
# 05. DEPENDENCY MANAGEMENT
# 05.1 Inspect Existing Environment
# ============================================================

import importlib
from importlib.metadata import version, PackageNotFoundError


REQUIRED_PACKAGES = [
    "torch",
    "transformers",
    "datasets",
    "accelerate",
    "evaluate",
    "jiwer",
    "soundfile",
    "librosa",
]


def get_package_version(package_name: str) -> str:
    """Return installed package version or NOT INSTALLED."""
    try:
        return version(package_name)
    except PackageNotFoundError:
        return "NOT INSTALLED"


print("=" * 70)
print("INSTALLED DEPENDENCY STATUS")
print("=" * 70)

for package in REQUIRED_PACKAGES:
    print(f"{package:<15} : {get_package_version(package)}")

print("=" * 70)

INSTALLED DEPENDENCY STATUS
torch           : 2.11.0+cu130
transformers    : 4.48.2
datasets        : 4.8.5
accelerate      : 1.14.0
evaluate        : 0.4.6
jiwer           : 4.0.0
soundfile       : 0.14.0
librosa         : 0.11.0


In [15]:
# ================================================================
# 05.2 Dependency Installation
# ================================================================

%pip install -q \
    "transformers==4.48.2" \
    "huggingface-hub==0.36.2" \
    "datasets==4.8.5" \
    "accelerate==1.14.0" \
    "evaluate==0.4.6" \
    "jiwer==4.0.0" \
    "soundfile==0.14.0" \
    "librosa==0.11.0"

In [16]:
# ============================================================
# 05.3 Dependency Verification
# ============================================================

import sys
import torch
import transformers
import datasets
import accelerate
import evaluate
import jiwer
import soundfile
import librosa
from importlib.metadata import version


print("=" * 70)
print("DEPENDENCY VERIFICATION")
print("=" * 70)

print(f"Python        : {sys.version.split()[0]}")
print(f"PyTorch       : {torch.__version__}")
print(f"Transformers  : {transformers.__version__}")
print(f"Datasets      : {datasets.__version__}")
print(f"Accelerate    : {accelerate.__version__}")
print(f"Evaluate      : {evaluate.__version__}")
print(f"JiWER         : {version('jiwer')}")
print(f"SoundFile     : {soundfile.__version__}")
print(f"Librosa       : {librosa.__version__}")

print("=" * 70)

assert torch.cuda.is_available(), "CUDA GPU is not available."

assert transformers.__version__ == "4.48.2", (
    f"Unexpected Transformers version: {transformers.__version__}"
)

print("[PASS] CUDA is available.")
print("[PASS] Transformers version is pinned to 4.48.2.")
print("[PASS] Required ASR dependencies imported successfully.")

print("=" * 70)

DEPENDENCY VERIFICATION
Python        : 3.13.15
PyTorch       : 2.11.0+cu130
Transformers  : 4.48.2
Datasets      : 4.8.5
Accelerate    : 1.14.0
Evaluate      : 0.4.6
JiWER         : 4.0.0
SoundFile     : 0.14.0
Librosa       : 0.11.0
[PASS] CUDA is available.
[PASS] Transformers version is pinned to 4.48.2.
[PASS] Required ASR dependencies imported successfully.


In [17]:
# ============================================================
# 05.4 Record Experiment Environment
# ============================================================

import json
import platform
import sys
import torch
from importlib.metadata import version


# ------------------------------------------------------------
# Package version helper
# ------------------------------------------------------------

def get_package_version(package_name: str) -> str:
    """Return the installed package version."""
    return version(package_name)


# ------------------------------------------------------------
# Environment information
# ------------------------------------------------------------

environment_info = {
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),

    "pytorch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "datasets_version": datasets.__version__,
    "accelerate_version": accelerate.__version__,
    "evaluate_version": evaluate.__version__,

    # JiWER does not expose __version__ reliably.
    "jiwer_version": get_package_version("jiwer"),

    "soundfile_version": soundfile.__version__,
    "librosa_version": librosa.__version__,

    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,

    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else "No GPU"
    ),
}


# ------------------------------------------------------------
# Save environment record
# ------------------------------------------------------------

environment_file = PROJECT_ROOT / "environment.json"

with open(environment_file, "w") as f:
    json.dump(environment_info, f, indent=4)


# ------------------------------------------------------------
# Display environment
# ------------------------------------------------------------

print("=" * 70)
print("EXPERIMENT ENVIRONMENT RECORDED")
print("=" * 70)

for key, value in environment_info.items():
    print(f"{key:<25}: {value}")

print("-" * 70)
print(f"Saved to: {environment_file}")
print("=" * 70)

EXPERIMENT ENVIRONMENT RECORDED
python_version           : 3.13.15
platform                 : Linux-6.6.122+-x86_64-with-glibc2.39
pytorch_version          : 2.11.0+cu130
transformers_version     : 4.48.2
datasets_version         : 4.8.5
accelerate_version       : 1.14.0
evaluate_version         : 0.4.6
jiwer_version            : 4.0.0
soundfile_version        : 0.14.0
librosa_version          : 0.11.0
cuda_available           : True
cuda_version             : 13.0
gpu                      : Tesla T4
----------------------------------------------------------------------
Saved to: /content/multilingual_kenyan_asr/environment.json


# 06. Google Drive & Dataset Access

This section connects the Colab runtime to Google Drive and retrieves the
verified multilingual Kenyan ASR dataset.

The dataset contains:

- Swahili
- Kikuyu
- Kalenjin

The original source datasets are not modified.
The portable dataset ZIP is used as the transfer artifact.

In [18]:
# ============================================================
# 06.1 Mount Google Drive
# ============================================================

from google.colab import drive

DRIVE_MOUNT = "/content/drive"

drive.mount(DRIVE_MOUNT)

print("=" * 70)
print("GOOGLE DRIVE")
print("=" * 70)
print(f"Mount point: {DRIVE_MOUNT}")
print("[PASS] Google Drive mounted successfully.")
print("=" * 70)

Mounted at /content/drive
GOOGLE DRIVE
Mount point: /content/drive
[PASS] Google Drive mounted successfully.


In [19]:
# ============================================================
# 06.2 Locate Dataset Archive
# ============================================================

from pathlib import Path


DRIVE_ROOT = Path("/content/drive/MyDrive")

DATASET_ARCHIVE_NAME = "colab_dataset.zip"


print("=" * 70)
print("SEARCHING GOOGLE DRIVE FOR DATASET ARCHIVE")
print("=" * 70)


matches = list(DRIVE_ROOT.rglob(DATASET_ARCHIVE_NAME))


if len(matches) == 0:

    raise FileNotFoundError(
        f"Could not find '{DATASET_ARCHIVE_NAME}' in Google Drive."
    )


if len(matches) > 1:

    print("Multiple matching files found:")

    for match in matches:
        print(f"  - {match}")

    raise RuntimeError(
        "Multiple dataset archives found. Keep only the intended archive "
        "or specify its exact path."
    )


DATASET_ARCHIVE = matches[0]

print(f"Dataset archive found:")
print(f"  {DATASET_ARCHIVE}")

print("=" * 70)

SEARCHING GOOGLE DRIVE FOR DATASET ARCHIVE
Dataset archive found:
  /content/drive/MyDrive/Final_Year_Project/ASR_Dataset/colab_dataset.zip


In [20]:
# ============================================================
# 06.3 Verify Dataset Archive
# ============================================================

archive_size_mb = DATASET_ARCHIVE.stat().st_size / (1024 ** 2)

print("=" * 70)
print("DATASET ARCHIVE VERIFICATION")
print("=" * 70)

print(f"Archive : {DATASET_ARCHIVE}")
print(f"Size    : {archive_size_mb:.2f} MB")

if archive_size_mb < 400:
    raise RuntimeError(
        "Dataset archive appears unexpectedly small. "
        "Check that the correct colab_dataset.zip was uploaded."
    )

print("[PASS] Dataset archive exists.")
print("[PASS] Archive size is within the expected range.")
print("=" * 70)

DATASET ARCHIVE VERIFICATION
Archive : /content/drive/MyDrive/Final_Year_Project/ASR_Dataset/colab_dataset.zip
Size    : 429.85 MB
[PASS] Dataset archive exists.
[PASS] Archive size is within the expected range.


In [21]:
# ============================================================
# 06.4 Inspect Dataset Archive
# ============================================================

import zipfile


with zipfile.ZipFile(DATASET_ARCHIVE, "r") as zip_ref:

    members = zip_ref.namelist()

    print("=" * 70)
    print("DATASET ARCHIVE CONTENTS")
    print("=" * 70)

    print(f"Total archive entries: {len(members)}")

    print("\nFirst 20 entries:")

    for member in members[:20]:
        print(f"  {member}")

    print("=" * 70)

DATASET ARCHIVE CONTENTS
Total archive entries: 2793

First 20 entries:
  colab_dataset/train.csv
  colab_dataset/validation.csv
  colab_dataset/audio/kalenjin/train/00000496.flac
  colab_dataset/audio/kalenjin/train/00000498.flac
  colab_dataset/audio/kalenjin/train/00000512.flac
  colab_dataset/audio/kalenjin/train/00000647.flac
  colab_dataset/audio/kalenjin/train/00000854.flac
  colab_dataset/audio/kalenjin/train/00000859.flac
  colab_dataset/audio/kalenjin/train/00000954.flac
  colab_dataset/audio/kalenjin/train/00000960.flac
  colab_dataset/audio/kalenjin/train/00001008.flac
  colab_dataset/audio/kalenjin/train/00001291.flac
  colab_dataset/audio/kalenjin/train/00001395.flac
  colab_dataset/audio/kalenjin/train/00001641.flac
  colab_dataset/audio/kalenjin/train/00001698.flac
  colab_dataset/audio/kalenjin/train/00001779.flac
  colab_dataset/audio/kalenjin/train/00001953.flac
  colab_dataset/audio/kalenjin/train/00002089.flac
  colab_dataset/audio/kalenjin/train/00002184.flac
  co

In [22]:
# ============================================================
# 06.5 Extract Dataset Archive
# ============================================================

import shutil
import zipfile
from pathlib import Path


# ------------------------------------------------------------
# Extraction locations
# ------------------------------------------------------------

EXTRACTION_ROOT = PROJECT_ROOT / "dataset"
RAW_EXTRACTION_ROOT = PROJECT_ROOT / "_raw_zip_extraction"


# ------------------------------------------------------------
# Clean previous extraction attempts
# ------------------------------------------------------------

for path in [EXTRACTION_ROOT, RAW_EXTRACTION_ROOT]:

    if path.exists():

        if path.is_dir():
            shutil.rmtree(path)

        else:
            path.unlink()


RAW_EXTRACTION_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 70)
print("EXTRACTING DATASET")
print("=" * 70)


# ------------------------------------------------------------
# Extract files only
# ------------------------------------------------------------

with zipfile.ZipFile(DATASET_ARCHIVE, "r") as zip_ref:

    file_count = 0

    for member in zip_ref.infolist():

        # Normalize Windows path separators
        archive_path = member.filename.replace("\\", "/")

        # Ignore everything outside colab_dataset/
        if not archive_path.startswith("colab_dataset/"):
            continue

        # Ignore the root directory and all explicit directories
        if member.is_dir():
            continue

        # Remove archive root
        relative_path = archive_path[len("colab_dataset/"):]

        if not relative_path:
            continue

        target_path = RAW_EXTRACTION_ROOT / relative_path

        # ----------------------------------------------------
        # Create parent directories safely
        # ----------------------------------------------------

        parent = target_path.parent

        if parent.exists() and not parent.is_dir():

            raise RuntimeError(
                f"PATH COLLISION DETECTED:\n"
                f"Expected directory:\n{parent}\n"
                f"but a file already exists there."
            )

        parent.mkdir(
            parents=True,
            exist_ok=True
        )

        # ----------------------------------------------------
        # Write file
        # ----------------------------------------------------

        with zip_ref.open(member) as source:

            with open(target_path, "wb") as destination:
                shutil.copyfileobj(
                    source,
                    destination
                )

        file_count += 1


print(f"Files extracted: {file_count:,}")


# ------------------------------------------------------------
# Verify extraction
# ------------------------------------------------------------

required_files = [
    RAW_EXTRACTION_ROOT / "train.csv",
    RAW_EXTRACTION_ROOT / "validation.csv",
]

required_directories = [
    RAW_EXTRACTION_ROOT / "audio",
    RAW_EXTRACTION_ROOT / "audio" / "swahili",
    RAW_EXTRACTION_ROOT / "audio" / "kikuyu",
    RAW_EXTRACTION_ROOT / "audio" / "kalenjin",
]


print("\nVERIFYING EXTRACTION")
print("-" * 70)


for path in required_files:

    if not path.is_file():
        raise FileNotFoundError(
            f"Required file missing: {path}"
        )

    print(f"[PASS] {path.relative_to(RAW_EXTRACTION_ROOT)}")


for path in required_directories:

    if not path.is_dir():
        raise FileNotFoundError(
            f"Required directory missing: {path}"
        )

    print(f"[PASS] {path.relative_to(RAW_EXTRACTION_ROOT)}")


# ------------------------------------------------------------
# Move verified extraction into final location
# ------------------------------------------------------------

shutil.move(
    str(RAW_EXTRACTION_ROOT),
    str(EXTRACTION_ROOT)
)


print("\n" + "=" * 70)
print("DATASET EXTRACTION COMPLETE")
print("=" * 70)

print(f"Dataset location: {EXTRACTION_ROOT}")
print(f"Files extracted : {file_count:,}")

print("[PASS] Dataset extraction completed successfully.")
print("=" * 70)

EXTRACTING DATASET
Files extracted: 2,793

VERIFYING EXTRACTION
----------------------------------------------------------------------
[PASS] train.csv
[PASS] validation.csv
[PASS] audio
[PASS] audio/swahili
[PASS] audio/kikuyu
[PASS] audio/kalenjin

DATASET EXTRACTION COMPLETE
Dataset location: /content/multilingual_kenyan_asr/dataset
Files extracted : 2,793
[PASS] Dataset extraction completed successfully.


# 07. Dataset Integrity Verification

This section verifies that the extracted Colab dataset matches the
final validated multilingual Kenyan ASR dataset.

The verification checks:

- manifest structure
- language distribution
- transcription integrity
- train/validation separation
- audio-file existence
- audio format and readability
- expected dataset counts

In [23]:
# ============================================================
# 07.1 DATASET INTEGRITY VERIFICATION
# ============================================================

import pandas as pd
from pathlib import Path

print("=" * 70)
print("DATASET INTEGRITY VERIFICATION")
print("=" * 70)


# ------------------------------------------------------------
# 1. Verify manifest files
# ------------------------------------------------------------

if not TRAIN_MANIFEST.exists():
    raise FileNotFoundError(
        f"Training manifest not found: {TRAIN_MANIFEST}"
    )

if not VALIDATION_MANIFEST.exists():
    raise FileNotFoundError(
        f"Validation manifest not found: {VALIDATION_MANIFEST}"
    )

train_df = pd.read_csv(TRAIN_MANIFEST)
validation_df = pd.read_csv(VALIDATION_MANIFEST)

print("\n[1] MANIFEST FILES")
print(f"Train manifest      : {TRAIN_MANIFEST}")
print(f"Validation manifest : {VALIDATION_MANIFEST}")
print("[PASS] Both manifest files exist.")


# ------------------------------------------------------------
# 2. Verify required columns
# ------------------------------------------------------------

required_columns = {
    "audio",
    "text",
    "language",
}

missing_train_columns = required_columns - set(train_df.columns)
missing_validation_columns = (
    required_columns - set(validation_df.columns)
)

if missing_train_columns:
    raise ValueError(
        f"Missing training columns: {sorted(missing_train_columns)}"
    )

if missing_validation_columns:
    raise ValueError(
        f"Missing validation columns: "
        f"{sorted(missing_validation_columns)}"
    )

print("\n[2] MANIFEST STRUCTURE")
print(f"Train columns      : {list(train_df.columns)}")
print(f"Validation columns : {list(validation_df.columns)}")
print("[PASS] Required manifest columns are present.")


# ------------------------------------------------------------
# 3. Verify expected language names
# ------------------------------------------------------------

EXPECTED_TRAIN_COUNTS = {
    "swahili": 1243,
    "kikuyu": 398,
    "kalenjin": 642,
}

EXPECTED_VALIDATION_COUNTS = {
    "swahili": 310,
    "kikuyu": 100,
    "kalenjin": 98,
}

actual_train_counts = train_df["language"].value_counts().to_dict()
actual_validation_counts = (
    validation_df["language"].value_counts().to_dict()
)

print("\n[3] LANGUAGE DISTRIBUTION")

print("Training:")
for language, expected in EXPECTED_TRAIN_COUNTS.items():
    actual = actual_train_counts.get(language, 0)
    print(f"  {language:<10}: {actual:>4} / {expected}")

print("Validation:")
for language, expected in EXPECTED_VALIDATION_COUNTS.items():
    actual = actual_validation_counts.get(language, 0)
    print(f"  {language:<10}: {actual:>4} / {expected}")

if actual_train_counts != EXPECTED_TRAIN_COUNTS:
    raise ValueError(
        "Training language counts do not match.\n"
        f"Expected: {EXPECTED_TRAIN_COUNTS}\n"
        f"Actual:   {actual_train_counts}"
    )

if actual_validation_counts != EXPECTED_VALIDATION_COUNTS:
    raise ValueError(
        "Validation language counts do not match.\n"
        f"Expected: {EXPECTED_VALIDATION_COUNTS}\n"
        f"Actual:   {actual_validation_counts}"
    )

print("[PASS] Language counts match the validated dataset.")


# ------------------------------------------------------------
# 4. Map dataset language names to Paza language codes
# ------------------------------------------------------------

LANGUAGE_CODES = {
    "swahili": "sw",
    "kikuyu": "kik",
    "kalenjin": "kln",
}

actual_languages = (
    set(train_df["language"].unique())
    | set(validation_df["language"].unique())
)

expected_languages = set(LANGUAGE_CODES.keys())

print("\n[4] LANGUAGE IDENTIFIERS")
print(f"Dataset language names : {sorted(actual_languages)}")
print(f"Paza language codes    : {LANGUAGE_CODES}")

if actual_languages != expected_languages:
    raise ValueError(
        f"Unexpected language names: {sorted(actual_languages)}"
    )

print("[PASS] Dataset language names are valid.")
print("[PASS] Paza language-code mapping established.")


# ------------------------------------------------------------
# 5. Verify total row counts
# ------------------------------------------------------------

EXPECTED_TRAIN_TOTAL = 2283
EXPECTED_VALIDATION_TOTAL = 508

if len(train_df) != EXPECTED_TRAIN_TOTAL:
    raise ValueError(
        f"Expected {EXPECTED_TRAIN_TOTAL} training rows, "
        f"found {len(train_df)}."
    )

if len(validation_df) != EXPECTED_VALIDATION_TOTAL:
    raise ValueError(
        f"Expected {EXPECTED_VALIDATION_TOTAL} validation rows, "
        f"found {len(validation_df)}."
    )

print("\n[5] TOTAL SAMPLE COUNTS")
print(f"Training samples     : {len(train_df)}")
print(f"Validation samples   : {len(validation_df)}")
print(f"Total samples        : {len(train_df) + len(validation_df)}")
print("[PASS] Total sample counts are correct.")


# ------------------------------------------------------------
# 6. Verify transcription integrity
# ------------------------------------------------------------

train_text = train_df["text"].fillna("").astype(str)
validation_text = validation_df["text"].fillna("").astype(str)

empty_train = (train_text.str.strip() == "").sum()
empty_validation = (validation_text.str.strip() == "").sum()

replacement_chars = (
    train_text.str.contains("�", regex=False).sum()
    + validation_text.str.contains("�", regex=False).sum()
)

codeswitch_markers = (
    train_text.str.contains(
        r"\[(cs|sc)\]",
        regex=True
    ).sum()
    + validation_text.str.contains(
        r"\[(cs|sc)\]",
        regex=True
    ).sum()
)

print("\n[6] TRANSCRIPTION INTEGRITY")
print(f"Empty training transcripts       : {empty_train}")
print(f"Empty validation transcripts     : {empty_validation}")
print(f"Replacement characters (�)       : {replacement_chars}")
print(f"Code-switch markers ([cs]/[sc])  : {codeswitch_markers}")

if empty_train or empty_validation:
    raise ValueError("Empty transcription(s) detected.")

if replacement_chars:
    raise ValueError("Unicode replacement character detected.")

if codeswitch_markers:
    raise ValueError("Unexpected [cs] or [sc] marker detected.")

print("[PASS] Transcriptions are clean.")


# ------------------------------------------------------------
# 7. Verify train/validation transcription leakage
# ------------------------------------------------------------

train_transcriptions = set(train_text)
validation_transcriptions = set(validation_text)

overlap = train_transcriptions.intersection(
    validation_transcriptions
)

print("\n[7] TRAIN / VALIDATION LEAKAGE")
print(f"Unique train transcriptions      : {len(train_transcriptions)}")
print(
    f"Unique validation transcriptions : "
    f"{len(validation_transcriptions)}"
)
print(f"Shared transcriptions            : {len(overlap)}")

if overlap:
    sample_overlap = sorted(overlap)[:5]

    raise ValueError(
        "Transcription leakage detected.\n"
        f"Examples: {sample_overlap}"
    )

print("[PASS] No train/validation transcription leakage.")


# ------------------------------------------------------------
# 8. Verify audio paths
# ------------------------------------------------------------

print("\n[8] AUDIO FILE VERIFICATION")


def resolve_audio_path(audio_path):
    """
    Resolve a manifest audio path.

    The portable manifest should contain paths relative
    to the dataset root. Absolute paths are also supported.
    """

    path = Path(str(audio_path))

    if path.is_absolute():
        return path

    return DATASET_ROOT / path


def verify_audio_paths(df, split_name):

    missing = []

    for audio_path in df["audio"]:

        path = resolve_audio_path(audio_path)

        if not path.exists():
            missing.append(str(path))

    print(
        f"{split_name:<12}: "
        f"{len(df):>4} rows | "
        f"{len(missing):>4} missing"
    )

    if missing:

        print("First missing files:")

        for path in missing[:5]:
            print(f"  {path}")

    return missing


train_missing = verify_audio_paths(
    train_df,
    "Training"
)

validation_missing = verify_audio_paths(
    validation_df,
    "Validation"
)

if train_missing or validation_missing:
    raise FileNotFoundError(
        "One or more manifest audio files are missing."
    )

print("[PASS] All manifest audio files exist.")


# ------------------------------------------------------------
# 9. Verify audio extensions
# ------------------------------------------------------------

audio_extensions = set()

for audio_path in pd.concat(
    [train_df["audio"], validation_df["audio"]],
    ignore_index=True
):

    audio_extensions.add(
        Path(str(audio_path)).suffix.lower()
    )

print("\n[9] AUDIO FORMATS")
print(
    f"Audio extensions found: "
    f"{sorted(audio_extensions)}"
)

EXPECTED_AUDIO_EXTENSIONS = {
    ".flac",
    ".wav",
}

if not audio_extensions.issubset(
    EXPECTED_AUDIO_EXTENSIONS
):

    unexpected = (
        audio_extensions
        - EXPECTED_AUDIO_EXTENSIONS
    )

    raise ValueError(
        f"Unexpected audio extension(s): "
        f"{sorted(unexpected)}"
    )

print(
    "[PASS] Audio formats are expected "
    "(.flac / .wav)."
)


# ------------------------------------------------------------
# 10. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DATASET INTEGRITY VERIFICATION COMPLETE")
print("=" * 70)

print(f"Training samples      : {len(train_df)}")
print(f"Validation samples    : {len(validation_df)}")
print(
    f"Total samples         : "
    f"{len(train_df) + len(validation_df)}"
)

print(
    f"Languages             : "
    f"{sorted(actual_languages)}"
)

print(
    f"Language codes        : "
    f"{LANGUAGE_CODES}"
)

print(
    f"Audio formats         : "
    f"{sorted(audio_extensions)}"
)

print(
    "\n[PASS] Dataset integrity verification "
    "completed successfully."
)

print("=" * 70)

DATASET INTEGRITY VERIFICATION

[1] MANIFEST FILES
Train manifest      : /content/multilingual_kenyan_asr/dataset/train.csv
Validation manifest : /content/multilingual_kenyan_asr/dataset/validation.csv
[PASS] Both manifest files exist.

[2] MANIFEST STRUCTURE
Train columns      : ['audio', 'text', 'language', 'speaker_id', 'original_id', 'split', 'original_filename', 'metadata_line', 'key', 'shard', 'duration', 'dataset_id', 'source', 'gender', 'domain', 'snr', 'dbfs', 'clip_ratio', 'sil_ratio', 'has_disfluency', 'domain_group']
Validation columns : ['audio', 'text', 'language', 'speaker_id', 'original_id', 'split', 'original_filename', 'metadata_line', 'key', 'shard', 'duration', 'dataset_id', 'source', 'gender', 'domain', 'snr', 'dbfs', 'clip_ratio', 'sil_ratio', 'has_disfluency', 'domain_group']
[PASS] Required manifest columns are present.

[3] LANGUAGE DISTRIBUTION
Training:
  swahili   : 1243 / 1243
  kikuyu    :  398 / 398
  kalenjin  :  642 / 642
Validation:
  swahili   :  310 

/tmp/ipykernel_590/1299570638.py:190: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  train_text.str.contains(
/tmp/ipykernel_590/1299570638.py:194: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  + validation_text.str.contains(


Training    : 2283 rows |    0 missing
Validation  :  508 rows |    0 missing
[PASS] All manifest audio files exist.

[9] AUDIO FORMATS
Audio extensions found: ['.flac', '.wav']
[PASS] Audio formats are expected (.flac / .wav).

DATASET INTEGRITY VERIFICATION COMPLETE
Training samples      : 2283
Validation samples    : 508
Total samples         : 2791
Languages             : ['kalenjin', 'kikuyu', 'swahili']
Language codes        : {'swahili': 'sw', 'kikuyu': 'kik', 'kalenjin': 'kln'}
Audio formats         : ['.flac', '.wav']

[PASS] Dataset integrity verification completed successfully.


# 08. Paza Whisper Large-v3-Turbo — Model & Processor

This section loads and verifies the pretrained Paza Whisper model and
its associated processor.

The model will be used as the base multilingual ASR model for:

- Swahili
- Kikuyu
- Kalenjin

The processor is responsible for converting raw audio into model inputs
and converting model predictions back into text.

Model loading is verified before any fine-tuning configuration is applied.

In [24]:
import torch

from transformers import (
    AutoProcessor,
    AutoModelForSpeechSeq2Seq,
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [25]:
logger.info(f"Loading processor from: {MODEL_NAME}")

processor = AutoProcessor.from_pretrained(MODEL_NAME)

logger.info("Processor loaded successfully.")

print("Processor type:", type(processor).__name__)
print("Feature extractor:", type(processor.feature_extractor).__name__)
print("Tokenizer:", type(processor.tokenizer).__name__)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Processor type: WhisperProcessor
Feature extractor: WhisperFeatureExtractor
Tokenizer: WhisperTokenizer


In [26]:
LANGUAGE_CODES = {
    "swahili": "sw",
    "kikuyu": "kik",
    "kalenjin": "kln",
}

tokenizer = processor.tokenizer

for language_name, language_code in LANGUAGE_CODES.items():
    token = f"<|{language_code}|>"
    token_id = tokenizer.convert_tokens_to_ids(token)

    print(
        f"{language_name:10s} | "
        f"code={language_code:3s} | "
        f"token={token:8s} | "
        f"id={token_id}"
    )

swahili    | code=sw  | token=<|sw|>   | id=50318
kikuyu     | code=kik | token=<|kik|>  | id=51866
kalenjin   | code=kln | token=<|kln|>  | id=51870


In [27]:
import torch
from transformers import AutoModelForSpeechSeq2Seq

logger.info(f"Loading model: {MODEL_NAME}")

# Keep model weights in FP32. The Hugging Face Trainer will apply
# mixed-precision FP16 autocasting on the Tesla T4 during training.
# Loading the model itself as FP16 can cause:
# ValueError: Attempting to unscale FP16 gradients.
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_NAME,
    low_cpu_mem_usage=True,
)

logger.info("Model loaded successfully.")

print("Model type:", type(model).__name__)
print("Model parameters:", f"{model.num_parameters():,}")
print("Model dtype:", model.dtype)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.24G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

Model type: WhisperForConditionalGeneration
Model parameters: 808,884,480
Model dtype: torch.float32


In [28]:
config = model.config

print("Model configuration")
print("-" * 60)

print("Architecture:", config.architectures)
print("Model type:", config.model_type)
print("Vocab size:", config.vocab_size)
print("Decoder start token:", config.decoder_start_token_id)
print("Max target positions:", config.max_target_positions)
print("EOS token:", config.eos_token_id)
print("PAD token:", config.pad_token_id)

Model configuration
------------------------------------------------------------
Architecture: ['WhisperForConditionalGeneration']
Model type: whisper
Vocab size: 51871
Decoder start token: 50258
Max target positions: 448
EOS token: 50257
PAD token: 50257


In [29]:
special_tokens = {
    "bos": tokenizer.bos_token,
    "eos": tokenizer.eos_token,
    "pad": tokenizer.pad_token,
    "unk": tokenizer.unk_token,
}

for name, token in special_tokens.items():
    print(f"{name:5s}: {token}")

bos  : <|endoftext|>
eos  : <|endoftext|>
pad  : <|endoftext|>
unk  : <|endoftext|>


In [30]:
print("\nLanguage token verification")
print("-" * 60)

for language_name, language_code in LANGUAGE_CODES.items():
    token = f"<|{language_code}|>"
    token_id = tokenizer.convert_tokens_to_ids(token)

    if token_id == tokenizer.unk_token_id:
        raise ValueError(
            f"{language_name} token {token} was not found in the tokenizer."
        )

    print(
        f"[PASS] {language_name}: "
        f"{token} -> token ID {token_id}"
    )


Language token verification
------------------------------------------------------------
[PASS] swahili: <|sw|> -> token ID 50318
[PASS] kikuyu: <|kik|> -> token ID 51866
[PASS] kalenjin: <|kln|> -> token ID 51870


In [31]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(DEVICE)

print("Device:", DEVICE)
print("Model device:", next(model.parameters()).device)

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved = torch.cuda.memory_reserved() / (1024 ** 3)

    print(f"GPU memory allocated: {allocated:.2f} GB")
    print(f"GPU memory reserved:   {reserved:.2f} GB")

# Gradient checkpointing requires the decoder cache to be disabled.
model.config.use_cache = False

print("[PASS] Model cache disabled for gradient-checkpointed training.")

Device: cuda
Model device: cuda:0
GPU memory allocated: 3.08 GB
GPU memory reserved:   3.15 GB
[PASS] Model cache disabled for gradient-checkpointed training.


# 09. Dataset Loading

This section loads the verified multilingual manifests into Hugging Face
Dataset objects.

The original audio files are not modified.

Audio remains referenced by path and is decoded only when required for
preprocessing.

The dataset contains:

- 2,283 training samples
- 508 validation samples

with:

- Swahili
- Kikuyu
- Kalenjin

In [32]:
from datasets import Dataset, DatasetDict
import pandas as pd

logger.info("Loading multilingual manifests...")

In [33]:
train_df = pd.read_csv(TRAIN_MANIFEST)
validation_df = pd.read_csv(VALIDATION_MANIFEST)

print("Training manifest shape:", train_df.shape)
print("Validation manifest shape:", validation_df.shape)

print("\nTraining language distribution:")
print(train_df["language"].value_counts())

print("\nValidation language distribution:")
print(validation_df["language"].value_counts())

Training manifest shape: (2283, 21)
Validation manifest shape: (508, 21)

Training language distribution:
language
swahili     1243
kalenjin     642
kikuyu       398
Name: count, dtype: int64

Validation language distribution:
language
swahili     310
kikuyu      100
kalenjin     98
Name: count, dtype: int64


In [34]:
print("Example training paths:")
for path in train_df["audio"].head(5):
    print(path)

print("\nExample validation paths:")
for path in validation_df["audio"].head(5):
    print(path)

Example training paths:
audio/swahili/train/1/audio_1.flac
audio/swahili/train/1/audio_3.flac
audio/swahili/train/1/audio_4.flac
audio/swahili/train/1/audio_6.flac
audio/swahili/train/1/audio_7.flac

Example validation paths:
audio/swahili/validation/1/audio_2.flac
audio/swahili/validation/1/audio_5.flac
audio/swahili/validation/1/audio_14.flac
audio/swahili/validation/1/audio_15.flac
audio/swahili/validation/1/audio_21.flac


In [35]:
def resolve_audio_path(relative_path):
    return str(DATASET_ROOT / relative_path)

train_df["audio_path"] = train_df["audio"].apply(resolve_audio_path)
validation_df["audio_path"] = validation_df["audio"].apply(resolve_audio_path)

In [36]:
print(train_df[["audio", "audio_path"]].head())

                                audio  \
0  audio/swahili/train/1/audio_1.flac   
1  audio/swahili/train/1/audio_3.flac   
2  audio/swahili/train/1/audio_4.flac   
3  audio/swahili/train/1/audio_6.flac   
4  audio/swahili/train/1/audio_7.flac   

                                          audio_path  
0  /content/multilingual_kenyan_asr/dataset/audio...  
1  /content/multilingual_kenyan_asr/dataset/audio...  
2  /content/multilingual_kenyan_asr/dataset/audio...  
3  /content/multilingual_kenyan_asr/dataset/audio...  
4  /content/multilingual_kenyan_asr/dataset/audio...  


In [37]:
missing_train = train_df.loc[
    ~train_df["audio_path"].map(lambda p: Path(p).exists()),
    "audio_path"
]

missing_validation = validation_df.loc[
    ~validation_df["audio_path"].map(lambda p: Path(p).exists()),
    "audio_path"
]

print("Missing training audio:", len(missing_train))
print("Missing validation audio:", len(missing_validation))

if len(missing_train) > 0:
    print("\nMissing training files:")
    print(missing_train.head())

if len(missing_validation) > 0:
    print("\nMissing validation files:")
    print(missing_validation.head())

assert len(missing_train) == 0, "Training audio files are missing."
assert len(missing_validation) == 0, "Validation audio files are missing."

print("\n[PASS] All manifest audio paths resolve successfully.")

Missing training audio: 0
Missing validation audio: 0

[PASS] All manifest audio paths resolve successfully.


In [38]:
train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

validation_dataset = Dataset.from_pandas(
    validation_df,
    preserve_index=False
)

dataset = DatasetDict({
    "train": train_dataset,
    "validation": validation_dataset,
})

In [39]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['audio', 'text', 'language', 'speaker_id', 'original_id', 'split', 'original_filename', 'metadata_line', 'key', 'shard', 'duration', 'dataset_id', 'source', 'gender', 'domain', 'snr', 'dbfs', 'clip_ratio', 'sil_ratio', 'has_disfluency', 'domain_group', 'audio_path'],
        num_rows: 2283
    })
    validation: Dataset({
        features: ['audio', 'text', 'language', 'speaker_id', 'original_id', 'split', 'original_filename', 'metadata_line', 'key', 'shard', 'duration', 'dataset_id', 'source', 'gender', 'domain', 'snr', 'dbfs', 'clip_ratio', 'sil_ratio', 'has_disfluency', 'domain_group', 'audio_path'],
        num_rows: 508
    })
})


In [40]:
for split in ["train", "validation"]:
    print(f"\n{split.upper()}")

    values = dataset[split].unique("language")

    print("Languages:", values)

    assert set(values) == {
        "swahili",
        "kikuyu",
        "kalenjin",
    }

print("\n[PASS] All three target languages are present in both splits.")


TRAIN
Languages: ['swahili', 'kikuyu', 'kalenjin']

VALIDATION
Languages: ['swahili', 'kikuyu', 'kalenjin']

[PASS] All three target languages are present in both splits.


In [41]:
LANGUAGE_CODES = {
    "swahili": "sw",
    "kikuyu": "kik",
    "kalenjin": "kln",
}

for language in dataset["train"].unique("language"):
    assert language in LANGUAGE_CODES

print(LANGUAGE_CODES)
print("[PASS] Every dataset language has a Paza language code.")

{'swahili': 'sw', 'kikuyu': 'kik', 'kalenjin': 'kln'}
[PASS] Every dataset language has a Paza language code.


In [42]:
example = dataset["train"][0]

print("Example fields:")
for key, value in example.items():
    print(f"{key}: {value}")

Example fields:
audio: audio/swahili/train/1/audio_1.flac
text: Habari za asubuhi?
language: swahili
speaker_id: 1
original_id: 1.0
split: train
original_filename: None
metadata_line: None
key: None
shard: None
duration: None
dataset_id: None
source: None
gender: None
domain: None
snr: None
dbfs: None
clip_ratio: None
sil_ratio: None
has_disfluency: None
domain_group: None
audio_path: /content/multilingual_kenyan_asr/dataset/audio/swahili/train/1/audio_1.flac


In [43]:
assert len(dataset["train"]) == 2283
assert len(dataset["validation"]) == 508

assert set(dataset["train"].unique("language")) == {
    "swahili",
    "kikuyu",
    "kalenjin",
}

assert set(dataset["validation"].unique("language")) == {
    "swahili",
    "kikuyu",
    "kalenjin",
}

print("=" * 70)
print("DATASET LOADING CHECKPOINT")
print("=" * 70)
print(f"Training samples   : {len(dataset['train'])}")
print(f"Validation samples : {len(dataset['validation'])}")
print(f"Total samples      : {len(dataset['train']) + len(dataset['validation'])}")
print("Languages          : Swahili, Kikuyu, Kalenjin")
print("[PASS] Dataset loading completed successfully.")

DATASET LOADING CHECKPOINT
Training samples   : 2283
Validation samples : 508
Total samples      : 2791
Languages          : Swahili, Kikuyu, Kalenjin
[PASS] Dataset loading completed successfully.


# 10. Audio Preprocessing

This section prepares the multilingual audio for Whisper.

The original audio files are not modified.

Audio is:

1. loaded from the verified manifest path
2. converted to mono when required
3. resampled to 16 kHz
4. converted to the format expected by the Whisper feature extractor

Audio quality and duration are also checked before training.

In [44]:
TARGET_SAMPLE_RATE = 16_000

print("Target sample rate:", TARGET_SAMPLE_RATE, "Hz")

Target sample rate: 16000 Hz


In [45]:
import soundfile as sf
import numpy as np

def inspect_audio(path):
    info = sf.info(path)

    return {
        "samplerate": info.samplerate,
        "channels": info.channels,
        "duration": info.duration,
        "format": info.format,
        "subtype": info.subtype,
    }

In [46]:
for language in ["swahili", "kikuyu", "kalenjin"]:
    row = train_df[train_df["language"] == language].iloc[0]
    path = row["audio_path"]

    info = inspect_audio(path)

    print(f"\n{language.upper()}")
    print("-" * 40)
    print("File:", path)
    print("Sample rate:", info["samplerate"])
    print("Channels:", info["channels"])
    print("Duration:", f"{info['duration']:.2f} s")
    print("Format:", info["format"])
    print("Subtype:", info["subtype"])


SWAHILI
----------------------------------------
File: /content/multilingual_kenyan_asr/dataset/audio/swahili/train/1/audio_1.flac
Sample rate: 44100
Channels: 1
Duration: 1.36 s
Format: FLAC
Subtype: PCM_24

KIKUYU
----------------------------------------
File: /content/multilingual_kenyan_asr/dataset/audio/kikuyu/train/111/GEN_001-1-1.wav
Sample rate: 24000
Channels: 1
Duration: 5.23 s
Format: WAV
Subtype: PCM_16

KALENJIN
----------------------------------------
File: /content/multilingual_kenyan_asr/dataset/audio/kalenjin/train/00061257.flac
Sample rate: 16000
Channels: 1
Duration: 9.76 s
Format: FLAC
Subtype: PCM_16


In [47]:
def load_audio(path):
    audio, sample_rate = sf.read(path, dtype="float32")

    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)

    return audio, sample_rate

In [48]:
for language in ["swahili", "kikuyu", "kalenjin"]:
    row = train_df[train_df["language"] == language].iloc[0]

    audio, sample_rate = load_audio(row["audio_path"])

    print(
        f"{language:10s} | "
        f"sample_rate={sample_rate:5d} | "
        f"samples={len(audio):7d} | "
        f"duration={len(audio)/sample_rate:.2f}s | "
        f"finite={np.isfinite(audio).all()}"
    )

swahili    | sample_rate=44100 | samples=  60062 | duration=1.36s | finite=True
kikuyu     | sample_rate=24000 | samples= 125439 | duration=5.23s | finite=True
kalenjin   | sample_rate=16000 | samples= 156160 | duration=9.76s | finite=True


In [49]:
import librosa

In [50]:
def resample_audio(audio, original_sr, target_sr=TARGET_SAMPLE_RATE):
    if original_sr == target_sr:
        return audio.astype(np.float32)

    return librosa.resample(
        audio,
        orig_sr=original_sr,
        target_sr=target_sr
    ).astype(np.float32)

In [51]:
for language in ["swahili", "kikuyu", "kalenjin"]:
    row = train_df[train_df["language"] == language].iloc[0]

    audio, original_sr = load_audio(row["audio_path"])
    audio_16k = resample_audio(audio, original_sr)

    duration = len(audio_16k) / TARGET_SAMPLE_RATE

    print(
        f"{language:10s} | "
        f"{original_sr:5d} Hz -> "
        f"{TARGET_SAMPLE_RATE:5d} Hz | "
        f"{len(audio_16k):7d} samples | "
        f"{duration:.2f}s"
    )

swahili    | 44100 Hz -> 16000 Hz |   21792 samples | 1.36s
kikuyu     | 24000 Hz -> 16000 Hz |   83626 samples | 5.23s
kalenjin   | 16000 Hz -> 16000 Hz |  156160 samples | 9.76s


In [52]:
def prepare_audio(path):
    audio, original_sr = load_audio(path)

    audio = resample_audio(
        audio,
        original_sr,
        TARGET_SAMPLE_RATE
    )

    return {
        "array": audio,
        "sampling_rate": TARGET_SAMPLE_RATE,
    }

In [53]:
test_path = train_df.iloc[0]["audio_path"]

processed_audio = prepare_audio(test_path)

print("Sampling rate:", processed_audio["sampling_rate"])
print("Number of samples:", len(processed_audio["array"]))
print("Duration:", f"{len(processed_audio['array']) / TARGET_SAMPLE_RATE:.2f}s")
print("dtype:", processed_audio["array"].dtype)
print("finite:", np.isfinite(processed_audio["array"]).all())

Sampling rate: 16000
Number of samples: 21792
Duration: 1.36s
dtype: float32
finite: True


In [54]:
print("Training duration statistics")
print("-" * 50)

print(
    train_df.groupby("language")["audio_path"]
    .apply(
        lambda paths: pd.Series(
            [sf.info(p).duration for p in paths]
        ).describe()
    )
)

Training duration statistics
--------------------------------------------------
language       
kalenjin  count     642.000000
          mean        6.198577
          std         4.751413
          min         1.120000
          25%         3.270406
          50%         4.640000
          75%         7.200000
          max        29.920000
kikuyu    count     398.000000
          mean        9.010064
          std         1.483814
          min         4.390708
          25%         8.002156
          50%         9.171125
          75%        10.033167
          max        15.336042
swahili   count    1243.000000
          mean        3.215201
          std         1.144923
          min         1.142721
          25%         2.428288
          50%         2.947415
          75%         3.656848
          max         7.555986
Name: audio_path, dtype: float64


In [55]:
duration_records = []

for _, row in train_df.iterrows():
    info = sf.info(row["audio_path"])

    duration_records.append({
        "language": row["language"],
        "duration": info.duration,
    })

duration_df = pd.DataFrame(duration_records)

print(
    duration_df.groupby("language")["duration"]
    .describe()
)

           count      mean       std       min       25%       50%        75%  \
language                                                                        
kalenjin   642.0  6.198577  4.751413  1.120000  3.270406  4.640000   7.200000   
kikuyu     398.0  9.010064  1.483814  4.390708  8.002156  9.171125  10.033167   
swahili   1243.0  3.215201  1.144923  1.142721  2.428288  2.947415   3.656848   

                max  
language             
kalenjin  29.920000  
kikuyu    15.336042  
swahili    7.555986  


In [56]:
print("=" * 70)
print("AUDIO SANITY CHECK")
print("=" * 70)

invalid_audio = []

for _, row in train_df.iterrows():
    path = row["audio_path"]

    try:
        audio, sr = load_audio(path)

        if len(audio) == 0:
            invalid_audio.append((path, "empty"))

        elif not np.isfinite(audio).all():
            invalid_audio.append((path, "non-finite"))

        elif sr <= 0:
            invalid_audio.append((path, "invalid sample rate"))

    except Exception as exc:
        invalid_audio.append((path, str(exc)))

print("Training files checked:", len(train_df))
print("Invalid training files:", len(invalid_audio))

assert len(invalid_audio) == 0

print("[PASS] All training audio files decode successfully.")

AUDIO SANITY CHECK
Training files checked: 2283
Invalid training files: 0
[PASS] All training audio files decode successfully.


In [57]:
invalid_audio_val = []

for _, row in validation_df.iterrows():
    path = row["audio_path"]

    try:
        audio, sr = load_audio(path)

        if len(audio) == 0:
            invalid_audio_val.append((path, "empty"))

        elif not np.isfinite(audio).all():
            invalid_audio_val.append((path, "non-finite"))

        elif sr <= 0:
            invalid_audio_val.append((path, "invalid sample rate"))

    except Exception as exc:
        invalid_audio_val.append((path, str(exc)))

print("Validation files checked:", len(validation_df))
print("Invalid validation files:", len(invalid_audio_val))

assert len(invalid_audio_val) == 0

print("[PASS] All validation audio files decode successfully.")

Validation files checked: 508
Invalid validation files: 0
[PASS] All validation audio files decode successfully.


# 11. Whisper Feature Extraction & Multilingual Labels

This section converts the verified 16 kHz audio into Whisper input
features and constructs language-aware decoder labels.

Each example retains its own language identity:

- Swahili → `sw`
- Kikuyu → `kik`
- Kalenjin → `kln`

No original audio files are modified.

In [58]:
feature_extractor = processor.feature_extractor
tokenizer = processor.tokenizer

print("Feature extractor:", type(feature_extractor).__name__)
print("Tokenizer:", type(tokenizer).__name__)
print("Sampling rate:", feature_extractor.sampling_rate)

Feature extractor: WhisperFeatureExtractor
Tokenizer: WhisperTokenizer
Sampling rate: 16000


In [59]:
assert feature_extractor.sampling_rate == TARGET_SAMPLE_RATE

print("[PASS] Whisper feature extractor expects 16 kHz audio.")

[PASS] Whisper feature extractor expects 16 kHz audio.


In [60]:
test_row = train_df.iloc[0]

audio = prepare_audio(test_row["audio_path"])

features = feature_extractor(
    audio["array"],
    sampling_rate=audio["sampling_rate"],
)

print("Feature keys:", features.keys())
print("Input features shape:", features["input_features"][0].shape)

Feature keys: dict_keys(['input_features'])
Input features shape: (128, 3000)


In [61]:
for language in ["swahili", "kikuyu", "kalenjin"]:
    row = train_df[train_df["language"] == language].iloc[0]

    audio = prepare_audio(row["audio_path"])

    features = feature_extractor(
        audio["array"],
        sampling_rate=TARGET_SAMPLE_RATE,
    )

    shape = features["input_features"][0].shape

    print(
        f"{language:10s} | "
        f"features={shape}"
    )

swahili    | features=(128, 3000)
kikuyu     | features=(128, 3000)
kalenjin   | features=(128, 3000)


In [62]:
# ============================================================
# 11.X Paza Multilingual Tokenization
# ============================================================

PAZA_LANGUAGE_TOKEN_IDS = {
    "sw": tokenizer.convert_tokens_to_ids("<|sw|>"),
    "kik": tokenizer.convert_tokens_to_ids("<|kik|>"),
    "kln": tokenizer.convert_tokens_to_ids("<|kln|>"),
}

TRANSCRIBE_TOKEN_ID = tokenizer.convert_tokens_to_ids("<|transcribe|>")
NO_TIMESTAMPS_TOKEN_ID = tokenizer.convert_tokens_to_ids("<|notimestamps|>")
BOS_TOKEN_ID = tokenizer.convert_tokens_to_ids("<|startoftranscript|>")
EOS_TOKEN_ID = tokenizer.eos_token_id

print("Paza language token IDs:")
for code, token_id in PAZA_LANGUAGE_TOKEN_IDS.items():
    print(f"{code:5s} -> {token_id}")

print("\nSpecial token IDs:")
print("BOS/startoftranscript:", BOS_TOKEN_ID)
print("transcribe:", TRANSCRIBE_TOKEN_ID)
print("notimestamps:", NO_TIMESTAMPS_TOKEN_ID)
print("EOS:", EOS_TOKEN_ID)

Paza language token IDs:
sw    -> 50318
kik   -> 51866
kln   -> 51870

Special token IDs:
BOS/startoftranscript: 50258
transcribe: 50360
notimestamps: 50364
EOS: 50257


In [63]:
# ============================================================
# Explicit Paza multilingual tokenizer
# ============================================================

def tokenize_transcription(text, language_code):
    """
    Tokenize a transcription using Paza-specific language tokens.

    We construct the Whisper decoder prefix explicitly because
    the standard WhisperTokenizer language validation does not
    recognize Paza's custom Kikuyu and Kalenjin language codes.
    """

    if language_code not in PAZA_LANGUAGE_TOKEN_IDS:
        raise ValueError(
            f"Unsupported Paza language code: {language_code}"
        )

    # Tokenize ONLY the transcription text.
    # add_special_tokens=False is essential here because we are
    # constructing the complete Whisper sequence ourselves.
    text_ids = tokenizer(
        text,
        add_special_tokens=False,
    ).input_ids

    # Whisper decoder prefix:
    # <|startoftranscript|>
    # <|language|>
    # <|transcribe|>
    # <|notimestamps|>
    prefix_ids = [
        BOS_TOKEN_ID,
        PAZA_LANGUAGE_TOKEN_IDS[language_code],
        TRANSCRIBE_TOKEN_ID,
        NO_TIMESTAMPS_TOKEN_ID,
    ]

    return prefix_ids + text_ids + [EOS_TOKEN_ID]

In [64]:
EXPECTED_LANGUAGE_TOKENS = {
    "swahili": "<|sw|>",
    "kikuyu": "<|kik|>",
    "kalenjin": "<|kln|>",
}

for language in ["swahili", "kikuyu", "kalenjin"]:

    language_code = LANGUAGE_CODES[language]

    row = train_df[
        train_df["language"] == language
    ].iloc[0]

    labels = tokenize_transcription(
        row["text"],
        language_code,
    )

    tokens = tokenizer.convert_ids_to_tokens(labels[:4])

    expected_token = EXPECTED_LANGUAGE_TOKENS[language]

    assert tokens[0] == "<|startoftranscript|>"
    assert tokens[1] == expected_token
    assert tokens[2] == "<|transcribe|>"
    assert tokens[3] == "<|notimestamps|>"

    print(f"[PASS] {language:10s} -> {tokens}")

[PASS] swahili    -> ['<|startoftranscript|>', '<|sw|>', '<|transcribe|>', '<|notimestamps|>']
[PASS] kikuyu     -> ['<|startoftranscript|>', '<|kik|>', '<|transcribe|>', '<|notimestamps|>']
[PASS] kalenjin   -> ['<|startoftranscript|>', '<|kln|>', '<|transcribe|>', '<|notimestamps|>']


In [65]:
def preprocess_example(example):
    audio = prepare_audio(example["audio_path"])

    input_features = feature_extractor(
        audio["array"],
        sampling_rate=TARGET_SAMPLE_RATE,
    ).input_features[0]

    language_name = example["language"]
    language_code = LANGUAGE_CODES[language_name]

    labels = tokenize_transcription(
        example["text"],
        language_code,
    )

    return {
        "input_features": input_features,
        "labels": labels,
        "language": language_name,
    }

In [66]:
for language in ["swahili", "kikuyu", "kalenjin"]:

    row = train_df[
        train_df["language"] == language
    ].iloc[0]

    processed = preprocess_example(row.to_dict())

    print(f"\n{language.upper()}")
    print("Text:", row["text"])
    print("Language:", processed["language"])
    print("Input features:", processed["input_features"].shape)
    print("Number of labels:", len(processed["labels"]))
    print(
        "First tokens:",
        tokenizer.convert_ids_to_tokens(
            processed["labels"][:6]
        )
    )


SWAHILI
Text: Habari za asubuhi?
Language: swahili
Input features: (128, 3000)
Number of labels: 14
First tokens: ['<|startoftranscript|>', '<|sw|>', '<|transcribe|>', '<|notimestamps|>', 'H', 'ab']

KIKUYU
Text: Kĩambĩrĩria-inĩ kĩa maũndũ mothe, Ngai nĩombire igũrũ na thĩ.
Language: kikuyu
Input features: (128, 3000)
Number of labels: 43
First tokens: ['<|startoftranscript|>', '<|kik|>', '<|transcribe|>', '<|notimestamps|>', 'K', 'Ä©']

KALENJIN
Text: Ngot ko sait taman ak aeng eng saitab kongasi eng Centaur,ko sait ata eng Kenya nguno
Language: kalenjin
Input features: (128, 3000)
Number of labels: 30
First tokens: ['<|startoftranscript|>', '<|kln|>', '<|transcribe|>', '<|notimestamps|>', 'N', 'got']


# 12. Label-Length Audit

Before full dataset preprocessing, the Whisper decoder label sequences are
audited to ensure that no transcription exceeds the model's maximum target
sequence length.

This is important because silently truncating transcripts would remove
information from the training targets and could be especially problematic
for medical speech, where symptoms, negations, numbers, medication names,
and other clinically relevant terms may occur near the end of an utterance.

The audit is performed:
- across the complete training set
- across the complete validation set
- separately for each language

No transcripts are modified during this audit.

In [67]:
# ============================================================
# 12.1 Label-Length Audit
# ============================================================

import numpy as np
import pandas as pd

MAX_LABEL_LENGTH = model.config.max_target_positions

print("=" * 70)
print("LABEL-LENGTH AUDIT")
print("=" * 70)

print(f"Maximum decoder target positions: {MAX_LABEL_LENGTH}")


def audit_label_lengths(df, split_name):
    records = []

    for _, row in df.iterrows():

        language_name = row["language"]
        language_code = LANGUAGE_CODES[language_name]

        labels = tokenize_transcription(
            row["text"],
            language_code,
        )

        records.append({
            "language": language_name,
            "text": row["text"],
            "label_length": len(labels),
        })

    audit_df = pd.DataFrame(records)

    print(f"\n{split_name.upper()}")
    print("-" * 70)

    print(f"Samples: {len(audit_df):,}")
    print(f"Minimum label length: {audit_df['label_length'].min()}")
    print(f"Maximum label length: {audit_df['label_length'].max()}")
    print(f"Mean label length: {audit_df['label_length'].mean():.2f}")
    print(f"Median label length: {audit_df['label_length'].median():.2f}")
    print(
        f"95th percentile: "
        f"{np.percentile(audit_df['label_length'], 95):.2f}"
    )
    print(
        f"99th percentile: "
        f"{np.percentile(audit_df['label_length'], 99):.2f}"
    )

    exceeding = audit_df[
        audit_df["label_length"] > MAX_LABEL_LENGTH
    ]

    print(
        f"Exceeding maximum ({MAX_LABEL_LENGTH}): "
        f"{len(exceeding)}"
    )

    print("\nPer-language statistics:")

    for language in ["swahili", "kikuyu", "kalenjin"]:

        language_df = audit_df[
            audit_df["language"] == language
        ]

        if len(language_df) == 0:
            continue

        print(
            f"{language:10s} | "
            f"n={len(language_df):4d} | "
            f"max={language_df['label_length'].max():3d} | "
            f"mean={language_df['label_length'].mean():6.2f}"
        )

    return audit_df


train_label_audit = audit_label_lengths(
    train_df,
    "training",
)

validation_label_audit = audit_label_lengths(
    validation_df,
    "validation",
)

LABEL-LENGTH AUDIT
Maximum decoder target positions: 448

TRAINING
----------------------------------------------------------------------
Samples: 2,283
Minimum label length: 8
Maximum label length: 133
Mean label length: 32.65
Median label length: 23.00
95th percentile: 86.00
99th percentile: 105.18
Exceeding maximum (448): 0

Per-language statistics:
swahili    | n=1243 | max= 43 | mean= 21.56
kikuyu     | n= 398 | max=125 | mean= 75.14
kalenjin   | n= 642 | max=133 | mean= 27.80

VALIDATION
----------------------------------------------------------------------
Samples: 508
Minimum label length: 10
Maximum label length: 143
Mean label length: 33.87
Median label length: 22.50
95th percentile: 90.00
99th percentile: 101.86
Exceeding maximum (448): 0

Per-language statistics:
swahili    | n= 310 | max= 43 | mean= 21.51
kikuyu     | n= 100 | max=110 | mean= 74.76
kalenjin   | n=  98 | max=143 | mean= 31.21


In [68]:
# ============================================================
# 12.2 Label-Length Verification
# ============================================================

train_over_limit = train_label_audit[
    train_label_audit["label_length"] > MAX_LABEL_LENGTH
]

validation_over_limit = validation_label_audit[
    validation_label_audit["label_length"] > MAX_LABEL_LENGTH
]

assert len(train_over_limit) == 0, (
    f"Training set contains {len(train_over_limit)} "
    f"samples exceeding {MAX_LABEL_LENGTH} tokens."
)

assert len(validation_over_limit) == 0, (
    f"Validation set contains {len(validation_over_limit)} "
    f"samples exceeding {MAX_LABEL_LENGTH} tokens."
)

print("=" * 70)
print("[PASS] No training labels exceed the model maximum.")
print("[PASS] No validation labels exceed the model maximum.")
print("[PASS] No transcription truncation is required.")
print("=" * 70)

[PASS] No training labels exceed the model maximum.
[PASS] No validation labels exceed the model maximum.
[PASS] No transcription truncation is required.


# 13. Full Dataset Preparation

The complete dataset is prepared without materializing Whisper input
features for every example in RAM.

Decoder labels are tokenized once and stored with the dataset.

Whisper input features are extracted dynamically at batch time by the
data collator. This avoids storing thousands of large feature matrices
simultaneously and keeps memory usage bounded by the training batch size.

The original audio and transcript data are not modified.

In [69]:
# ============================================================
# 13.1 Memory-Efficient Dataset Preparation
# ============================================================

from datasets import Dataset, DatasetDict
from tqdm.auto import tqdm


def prepare_labels_dataframe(df):
    """
    Add Paza multilingual decoder labels without extracting
    and storing Whisper audio features.
    """

    records = []

    for _, row in tqdm(
        df.iterrows(),
        total=len(df),
        desc="Tokenizing labels",
    ):

        language_name = row["language"]
        language_code = LANGUAGE_CODES[language_name]

        labels = tokenize_transcription(
            row["text"],
            language_code,
        )

        records.append({
            "audio_path": str(row["audio_path"]),
            "text":       row["text"],
            "language":   language_name,
            # ✅ FIX: speaker_id is absent in Kalenjin rows.
            #         Use .get() with a default so all three
            #         languages produce the same schema.
            "speaker_id": str(row.get("speaker_id", "unknown")),
            "labels":     labels,
        })

    return Dataset.from_list(records)


print("=" * 70)
print("PREPARING TRAINING DATASET")
print("=" * 70)

train_processed = prepare_labels_dataframe(train_df)

print("\n" + "=" * 70)
print("PREPARING VALIDATION DATASET")
print("=" * 70)

validation_processed = prepare_labels_dataframe(validation_df)

processed_dataset = DatasetDict({
    "train":      train_processed,
    "validation": validation_processed,
})

print("\nProcessed dataset:")
print(processed_dataset)

PREPARING TRAINING DATASET


Tokenizing labels:   0%|          | 0/2283 [00:00<?, ?it/s]


PREPARING VALIDATION DATASET


Tokenizing labels:   0%|          | 0/508 [00:00<?, ?it/s]


Processed dataset:
DatasetDict({
    train: Dataset({
        features: ['audio_path', 'text', 'language', 'speaker_id', 'labels'],
        num_rows: 2283
    })
    validation: Dataset({
        features: ['audio_path', 'text', 'language', 'speaker_id', 'labels'],
        num_rows: 508
    })
})


In [70]:
# ============================================================
# 13.2 Processed Dataset Verification
# ============================================================

print("=" * 70)
print("PROCESSED DATASET VERIFICATION")
print("=" * 70)

assert len(processed_dataset["train"]) == 2283
assert len(processed_dataset["validation"]) == 508

print(
    f"Training examples   : {len(processed_dataset['train']):,}"
)

print(
    f"Validation examples : {len(processed_dataset['validation']):,}"
)

print(
    f"Total examples      : "
    f"{len(processed_dataset['train']) + len(processed_dataset['validation']):,}"
)

required_columns = {
    "audio_path",
    "text",
    "language",
    "speaker_id",
    "labels",
}

for split in ["train", "validation"]:

    columns = set(
        processed_dataset[split].column_names
    )

    assert required_columns.issubset(columns)

    print(
        f"[PASS] {split}: required columns present."
    )

print("\n[PASS] Dataset sizes are correct.")

PROCESSED DATASET VERIFICATION
Training examples   : 2,283
Validation examples : 508
Total examples      : 2,791
[PASS] train: required columns present.
[PASS] validation: required columns present.

[PASS] Dataset sizes are correct.


In [71]:
# ============================================================
# 13.3 Label Verification
# ============================================================

sample = processed_dataset["train"][0]

print("=" * 70)
print("LABEL VERIFICATION")
print("=" * 70)

print("Language:")
print(sample["language"])

print("\nText:")
print(sample["text"])

print("\nNumber of labels:")
print(len(sample["labels"]))

print("\nFirst decoder tokens:")

print(
    tokenizer.convert_ids_to_tokens(
        sample["labels"][:10]
    )
)

# Verify the language token
language_code = LANGUAGE_CODES[sample["language"]]

expected_language_token = (
    tokenizer.convert_ids_to_tokens(
        [PAZA_LANGUAGE_TOKEN_IDS[language_code]]
    )[0]
)

actual_tokens = tokenizer.convert_ids_to_tokens(
    sample["labels"][:4]
)

print("\nDecoder prefix:")
print(actual_tokens)

assert actual_tokens[0] == "<|startoftranscript|>"
assert actual_tokens[1] == expected_language_token
assert actual_tokens[2] == "<|transcribe|>"
assert actual_tokens[3] == "<|notimestamps|>"

print("\n[PASS] Paza multilingual decoder labels are valid.")

LABEL VERIFICATION
Language:
swahili

Text:
Habari za asubuhi?

Number of labels:
14

First decoder tokens:
['<|startoftranscript|>', '<|sw|>', '<|transcribe|>', '<|notimestamps|>', 'H', 'ab', 'ari', 'Ġza', 'Ġas', 'ub']

Decoder prefix:
['<|startoftranscript|>', '<|sw|>', '<|transcribe|>', '<|notimestamps|>']

[PASS] Paza multilingual decoder labels are valid.


In [72]:
# ============================================================
# 13.4 Language Distribution Verification
# ============================================================

EXPECTED_COUNTS = {
    "train": {
        "swahili": 1243,
        "kikuyu": 398,
        "kalenjin": 642,
    },
    "validation": {
        "swahili": 310,
        "kikuyu": 100,
        "kalenjin": 98,
    },
}


print("=" * 70)
print("LANGUAGE DISTRIBUTION")
print("=" * 70)

for split in ["train", "validation"]:

    counts = {}

    for language in processed_dataset[split]["language"]:
        counts[language] = counts.get(language, 0) + 1

    print(f"\n{split.upper()}")

    for language, expected in EXPECTED_COUNTS[split].items():

        actual = counts.get(language, 0)

        print(
            f"{language:10s}: "
            f"{actual:4d} / {expected:4d}"
        )

        assert actual == expected


print("\n[PASS] Language distribution preserved.")

LANGUAGE DISTRIBUTION

TRAIN
swahili   : 1243 / 1243
kikuyu    :  398 /  398
kalenjin  :  642 /  642

VALIDATION
swahili   :  310 /  310
kikuyu    :  100 /  100
kalenjin  :   98 /   98

[PASS] Language distribution preserved.


In [73]:
print("=" * 70)
print("DYNAMIC FEATURE EXTRACTION TEST")
print("=" * 70)

EXPECTED_MEL_BINS = feature_extractor.feature_size
EXPECTED_FRAMES = feature_extractor.nb_max_frames

print(f"Expected mel bins : {EXPECTED_MEL_BINS}")
print(f"Expected frames   : {EXPECTED_FRAMES}")
print()

for language in ["swahili", "kikuyu", "kalenjin"]:

    row = train_df[
        train_df["language"] == language
    ].iloc[0]

    audio = prepare_audio(row["audio_path"])

    features = feature_extractor(
        audio["array"],
        sampling_rate=TARGET_SAMPLE_RATE,
    ).input_features[0]

    feature_shape = np.asarray(features).shape

    print(
        f"{language:10s} | "
        f"audio samples = {len(audio['array']):6d} | "
        f"features = {feature_shape}"
    )

    assert feature_shape[0] == EXPECTED_MEL_BINS
    assert feature_shape[1] == EXPECTED_FRAMES
    assert np.isfinite(features).all()

print()
print("[PASS] Dynamic Whisper feature extraction works.")

DYNAMIC FEATURE EXTRACTION TEST
Expected mel bins : 128
Expected frames   : 3000

swahili    | audio samples =  21792 | features = (128, 3000)
kikuyu     | audio samples =  83626 | features = (128, 3000)
kalenjin   | audio samples = 156160 | features = (128, 3000)

[PASS] Dynamic Whisper feature extraction works.


# 14. Memory-Efficient Data Collator

Whisper input features are extracted dynamically at batch time rather
than stored for the entire dataset.

This keeps RAM usage bounded and avoids materializing thousands of
large `(128, 3000)` feature matrices.

The collator performs:

1. Audio loading and resampling to 16 kHz.
2. Whisper feature extraction.
3. Label padding.
4. Conversion of label padding positions to `-100` so they are
   ignored by the loss function.

The original audio and transcript data remain unchanged.

In [74]:
from dataclasses import dataclass
from typing import Any, Dict, List

import torch


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:

    processor: Any
    feature_extractor: Any
    tokenizer: Any

    def __call__(
        self,
        features: List[Dict[str, Any]],
    ) -> Dict[str, torch.Tensor]:

        # --------------------------------------------------------------
        # 1. Load audio for the current batch
        # --------------------------------------------------------------

        audio_arrays = []

        for feature in features:
            audio = prepare_audio(
                feature["audio_path"]
            )

            audio_arrays.append(
                audio["array"]
            )

        # --------------------------------------------------------------
        # 2. Extract Whisper/Paza features dynamically
        # --------------------------------------------------------------

        batch = self.feature_extractor(
            audio_arrays,
            sampling_rate=TARGET_SAMPLE_RATE,
            return_tensors="pt",
        )

        # --------------------------------------------------------------
        # 3. Prepare labels
        # --------------------------------------------------------------

        label_features = [
            {"input_ids": feature["labels"]}
            for feature in features
        ]

        labels_batch = self.tokenizer.pad(
            label_features,
            padding=True,
            return_tensors="pt",
        )

        # --------------------------------------------------------------
        # 4. Replace padding tokens with -100
        # --------------------------------------------------------------

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100,
        )

        batch["labels"] = labels

        return batch

In [75]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)

print("[PASS] Data collator created successfully.")

[PASS] Data collator created successfully.


In [76]:
print("=" * 70)
print("DATA COLLATOR SANITY CHECK")
print("=" * 70)

test_examples = [
    processed_dataset["train"][i]
    for i in range(2)
]

batch = data_collator(test_examples)

print("\nBatch keys:")
print(batch.keys())

print("\nInput features shape:")
print(batch["input_features"].shape)

print("\nInput features dtype:")
print(batch["input_features"].dtype)

print("\nLabels shape:")
print(batch["labels"].shape)

print("\nLabels dtype:")
print(batch["labels"].dtype)

DATA COLLATOR SANITY CHECK

Batch keys:
dict_keys(['input_features', 'labels'])

Input features shape:
torch.Size([2, 128, 3000])

Input features dtype:
torch.float32

Labels shape:
torch.Size([2, 15])

Labels dtype:
torch.int64


In [77]:
print("=" * 70)
print("LABEL PADDING VERIFICATION")
print("=" * 70)

labels = batch["labels"]

print("Minimum label value:", labels.min().item())
print("Maximum label value:", labels.max().item())

padding_positions = (labels == -100).sum().item()

print("Number of -100 padding positions:", padding_positions)

assert labels.ndim == 2
assert labels.dtype == torch.long
assert torch.isfinite(labels.float()).all()

print("\n[PASS] Labels are correctly padded.")

LABEL PADDING VERIFICATION
Minimum label value: -100
Maximum label value: 50364
Number of -100 padding positions: 1

[PASS] Labels are correctly padded.


In [78]:
print("=" * 70)
print("Paza FEATURE SHAPE VERIFICATION")
print("=" * 70)

expected_mel_bins = feature_extractor.feature_size
expected_frames = feature_extractor.nb_max_frames

actual_shape = tuple(batch["input_features"].shape)

print("Expected:")
print(
    f"Batch × {expected_mel_bins} × {expected_frames}"
)

print("\nActual:")
print(actual_shape)

assert actual_shape[1] == expected_mel_bins
assert actual_shape[2] == expected_frames

assert torch.isfinite(
    batch["input_features"]
).all()

print("\n[PASS] Paza feature dimensions are correct.")

Paza FEATURE SHAPE VERIFICATION
Expected:
Batch × 128 × 3000

Actual:
(2, 128, 3000)

[PASS] Paza feature dimensions are correct.


In [79]:
print("=" * 70)
print("PART 14 COMPLETE")
print("=" * 70)

print(f"Training examples   : {len(processed_dataset['train']):,}")
print(f"Validation examples : {len(processed_dataset['validation']):,}")
print(
    f"Feature dimensions  : "
    f"{expected_mel_bins} × {expected_frames}"
)
print(
    f"Test batch size     : "
    f"{batch['input_features'].shape[0]}"
)

print("\n[PASS] Memory-efficient data collator is ready.")

PART 14 COMPLETE
Training examples   : 2,283
Validation examples : 508
Feature dimensions  : 128 × 3000
Test batch size     : 2

[PASS] Memory-efficient data collator is ready.


# 15. Training Configuration

The Paza Whisper model will be fine-tuned using the verified multilingual
training dataset.

Because the available GPU is a Tesla T4 with approximately 14.6 GB of
VRAM, the initial configuration uses a small per-device batch size and
gradient accumulation.

Training will first be validated with a short sanity run before full
fine-tuning.

The validation dataset remains completely separate from training.

In [80]:
import torch
from transformers import Seq2SeqTrainingArguments


TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1

GRADIENT_ACCUMULATION_STEPS = 8

LEARNING_RATE = 1e-5
NUM_TRAIN_EPOCHS = 3

WARMUP_RATIO = 0.05

LOGGING_STEPS = 10
EVAL_STEPS = 100
SAVE_STEPS = 100

OUTPUT_DIR = str(CHECKPOINT_ROOT)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # --------------------------------------------------------------
    # Batch configuration
    # --------------------------------------------------------------
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    # --------------------------------------------------------------
    # Optimization
    # --------------------------------------------------------------
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    warmup_ratio=WARMUP_RATIO,

    # --------------------------------------------------------------
    # Evaluation and checkpointing
    # --------------------------------------------------------------
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,

    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,

    # --------------------------------------------------------------
    # Logging
    # --------------------------------------------------------------
    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,

    # --------------------------------------------------------------
    # Precision
    # --------------------------------------------------------------
    # Keep model weights in FP32 and let Trainer manage AMP.
    fp16=torch.cuda.is_available(),

    # --------------------------------------------------------------
    # Memory efficiency
    # --------------------------------------------------------------
    gradient_checkpointing=True,

    # --------------------------------------------------------------
    # Performance
    # --------------------------------------------------------------
    dataloader_num_workers=2,

    # --------------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------------
    seed=SEED,
    data_seed=SEED,

    # --------------------------------------------------------------
    # Generation
    # --------------------------------------------------------------
    predict_with_generate=False,

    # --------------------------------------------------------------
    # Reporting
    # --------------------------------------------------------------
    report_to="none",

    # --------------------------------------------------------------
    # Checkpoint selection
    # --------------------------------------------------------------
    load_best_model_at_end=False,
)

In [81]:
print("=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

print(f"Model                  : {MODEL_NAME}")
print(f"Train batch size       : {TRAIN_BATCH_SIZE}")
print(f"Eval batch size        : {EVAL_BATCH_SIZE}")
print(f"Gradient accumulation  : {GRADIENT_ACCUMULATION_STEPS}")
print(f"Effective train batch  : "
      f"{TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")

print(f"Learning rate          : {LEARNING_RATE}")
print(f"Epochs                 : {NUM_TRAIN_EPOCHS}")
print(f"Warmup ratio           : {WARMUP_RATIO}")

print(f"FP16                   : {torch.cuda.is_available()}")
print(f"GPU                    : {torch.cuda.get_device_name(0)}")

print(f"Output directory       : {OUTPUT_DIR}")

TRAINING CONFIGURATION
Model                  : microsoft/paza-whisper-large-v3-turbo
Train batch size       : 1
Eval batch size        : 1
Gradient accumulation  : 8
Effective train batch  : 8
Learning rate          : 1e-05
Epochs                 : 3
Warmup ratio           : 0.05
FP16                   : True
GPU                    : Tesla T4
Output directory       : /content/multilingual_kenyan_asr/outputs/checkpoints


In [82]:
assert TRAIN_BATCH_SIZE == 1
assert EVAL_BATCH_SIZE == 1

assert GRADIENT_ACCUMULATION_STEPS >= 1
assert LEARNING_RATE > 0
assert NUM_TRAIN_EPOCHS >= 1

assert torch.cuda.is_available()

print("[PASS] Training configuration is valid.")

[PASS] Training configuration is valid.


# 16. Training Sanity Check

Before full fine-tuning, a short training run is used to verify the
complete training pipeline.

This sanity check verifies:

- Paza model forward pass
- Dynamic audio feature extraction
- Multilingual decoder labels
- Data collator
- Loss computation
- Backpropagation
- Optimizer updates
- GPU memory usage

Only a small subset of the training data is used.

The full training configuration is not modified by this test.

In [83]:
# ============================================================
# 16.1 Sanity Dataset
# ============================================================

# Use a small language-balanced subset so the sanity run exercises
# Swahili, Kikuyu, and Kalenjin decoder labels.
SANITY_EXAMPLES_PER_LANGUAGE = 2
SANITY_LANGUAGES = ["swahili", "kikuyu", "kalenjin"]

sanity_indices = []

for language in SANITY_LANGUAGES:
    language_indices = [
        i
        for i, value in enumerate(processed_dataset["train"]["language"])
        if value == language
    ]

    if len(language_indices) < SANITY_EXAMPLES_PER_LANGUAGE:
        raise ValueError(
            f"Not enough training examples for {language}: "
            f"found {len(language_indices)}"
        )

    sanity_indices.extend(
        language_indices[:SANITY_EXAMPLES_PER_LANGUAGE]
    )

sanity_train_dataset = processed_dataset["train"].select(sanity_indices)
SANITY_TRAIN_SIZE = len(sanity_indices)

print("=" * 70)
print("TRAINING SANITY DATASET")
print("=" * 70)

print(f"Examples: {SANITY_TRAIN_SIZE}")
print("\nLanguages:")

language_counts = {}
for language in sanity_train_dataset["language"]:
    language_counts[language] = language_counts.get(language, 0) + 1

for language in SANITY_LANGUAGES:
    count = language_counts.get(language, 0)
    print(f"  {language:10s}: {count}")
    assert count == SANITY_EXAMPLES_PER_LANGUAGE

assert len(sanity_train_dataset) == SANITY_TRAIN_SIZE
assert set(sanity_train_dataset["language"]) == set(SANITY_LANGUAGES)

print("\n[PASS] Language-balanced sanity dataset created.")

TRAINING SANITY DATASET
Examples: 6

Languages:
  swahili   : 2
  kikuyu    : 2
  kalenjin  : 2

[PASS] Language-balanced sanity dataset created.


In [84]:
# ============================================================
# 16.2 Sanity Training Configuration
# ============================================================

import torch
from transformers import Seq2SeqTrainingArguments

SANITY_OUTPUT_DIR = str(
    OUTPUT_ROOT / "sanity_check"
)

sanity_training_args = Seq2SeqTrainingArguments(
    output_dir=SANITY_OUTPUT_DIR,

    # ----------------------------------------------------------
    # Batch configuration
    # ----------------------------------------------------------
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,

    # ----------------------------------------------------------
    # Optimization
    # ----------------------------------------------------------
    learning_rate=LEARNING_RATE,
    max_steps=2,

    # ----------------------------------------------------------
    # Logging
    # ----------------------------------------------------------
    logging_steps=1,

    # ----------------------------------------------------------
    # Checkpointing and evaluation
    # ----------------------------------------------------------
    save_strategy="no",
    eval_strategy="no",

    # ----------------------------------------------------------
    # Precision
    # ----------------------------------------------------------
    # The model is stored in FP32; Trainer applies AMP safely.
    fp16=torch.cuda.is_available(),

    # Keep the short sanity run simple.
    gradient_checkpointing=False,

    # ----------------------------------------------------------
    # Data loading
    # ----------------------------------------------------------
    dataloader_num_workers=0,

    # ----------------------------------------------------------
    # ✅ FIX: Set remove_unused_columns=False directly in the
    #         constructor so the trainer always receives the
    #         correct value regardless of cell execution order.
    #
    #         The dataset contains "audio_path", "language", and
    #         "speaker_id" columns that the model does not consume
    #         directly. The data collator needs them to load and
    #         preprocess audio at batch time. Without this flag,
    #         Trainer drops those columns before calling the
    #         collator and the collator raises a KeyError.
    # ----------------------------------------------------------
    remove_unused_columns=False,

    # ----------------------------------------------------------
    # Generation
    # ----------------------------------------------------------
    predict_with_generate=False,

    # ----------------------------------------------------------
    # Reporting
    # ----------------------------------------------------------
    report_to="none",

    # ----------------------------------------------------------
    # Reproducibility
    # ----------------------------------------------------------
    seed=SEED,
    data_seed=SEED,
)

print("=" * 70)
print("SANITY TRAINING CONFIGURATION")
print("=" * 70)

print(f"Output dir             : {SANITY_OUTPUT_DIR}")
print(f"Max steps              : {sanity_training_args.max_steps}")
print(f"Batch size             : {sanity_training_args.per_device_train_batch_size}")
print(f"FP16                   : {sanity_training_args.fp16}")
print(f"remove_unused_columns  : {sanity_training_args.remove_unused_columns}")

assert sanity_training_args.remove_unused_columns is False

print("\n[PASS] Sanity training configuration created.")

SANITY TRAINING CONFIGURATION
Output dir             : /content/multilingual_kenyan_asr/outputs/sanity_check
Max steps              : 2
Batch size             : 1
FP16                   : True
remove_unused_columns  : False

[PASS] Sanity training configuration created.


In [85]:
# ============================================================
# 16.3 Sanity Trainer
# ============================================================

from transformers import Seq2SeqTrainer

sanity_trainer = Seq2SeqTrainer(
    model=model,
    args=sanity_training_args,
    train_dataset=sanity_train_dataset,
    data_collator=data_collator,
)

print("[PASS] Sanity trainer created.")

[PASS] Sanity trainer created.


In [86]:
# ============================================================
# 16.4 GPU Memory — Before Sanity Training
# ============================================================

import torch

if torch.cuda.is_available():

    torch.cuda.empty_cache()

    # ✅ FIX: Reset peak memory counter before the sanity run
    #         so max_memory_allocated() below correctly reports
    #         the peak for this run only, not since process start.
    torch.cuda.reset_peak_memory_stats()

    allocated_before = (
        torch.cuda.memory_allocated() / 1024 ** 3
    )

    reserved_before = (
        torch.cuda.memory_reserved() / 1024 ** 3
    )

    print("=" * 70)
    print("GPU MEMORY BEFORE SANITY TRAINING")
    print("=" * 70)

    print(f"Allocated : {allocated_before:.2f} GB")
    print(f"Reserved  : {reserved_before:.2f} GB")

else:

    # ✅ FIX: Explicit fallback message when no GPU is present.
    print("=" * 70)
    print("GPU MEMORY BEFORE SANITY TRAINING")
    print("=" * 70)
    print("CUDA not available — running on CPU.")

GPU MEMORY BEFORE SANITY TRAINING
Allocated : 3.08 GB
Reserved  : 3.15 GB


In [87]:
# ============================================================
# 16.5 Run Sanity Training
# ============================================================

print("=" * 70)
print("STARTING TRAINING SANITY CHECK")
print("=" * 70)

model.train()
model.config.use_cache = False

try:
    sanity_result = sanity_trainer.train()
except Exception:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    raise

print("\n" + "=" * 70)
print("SANITY TRAINING FINISHED")
print("=" * 70)

print(f"Training loss : {sanity_result.training_loss:.4f}")
print(f"Global steps  : {sanity_result.global_step}")

assert sanity_result.global_step == 2

STARTING TRAINING SANITY CHECK


Step,Training Loss
1,3.907300
2,1.516900



SANITY TRAINING FINISHED
Training loss : 2.7121
Global steps  : 2


In [88]:
# ============================================================
# 16.6 GPU Memory — After Sanity Training
# ============================================================

import torch

if torch.cuda.is_available():

    allocated_after = (
        torch.cuda.memory_allocated() / 1024 ** 3
    )

    reserved_after = (
        torch.cuda.memory_reserved() / 1024 ** 3
    )

    max_allocated = (
        torch.cuda.max_memory_allocated() / 1024 ** 3
    )

    print("=" * 70)
    print("GPU MEMORY AFTER SANITY TRAINING")
    print("=" * 70)

    print(f"Allocated       : {allocated_after:.2f} GB")
    print(f"Reserved        : {reserved_after:.2f} GB")
    print(f"Peak allocated  : {max_allocated:.2f} GB")

else:

    print("=" * 70)
    print("GPU MEMORY AFTER SANITY TRAINING")
    print("=" * 70)
    print("CUDA not available — running on CPU.")

GPU MEMORY AFTER SANITY TRAINING
Allocated       : 3.10 GB
Reserved        : 9.66 GB
Peak allocated  : 7.55 GB


In [89]:
# ============================================================
# 16.7 Sanity Check Verification
# ============================================================

import numpy as np

assert "sanity_result" in globals(), (
    "Sanity training did not produce a result."
)

assert sanity_result.training_loss is not None, (
    "Training loss is None — the sanity run did not complete."
)

assert np.isfinite(sanity_result.training_loss), (
    f"Training loss is not finite: {sanity_result.training_loss}"
)

assert sanity_result.global_step == 2, (
    f"Expected 2 sanity steps, got {sanity_result.global_step}."
)

print("=" * 70)
print("PART 16 COMPLETE")
print("=" * 70)

print("[PASS] Training sanity check completed.")
print(f"Sanity training loss : {sanity_result.training_loss:.4f}")
print(f"Sanity global steps  : {sanity_result.global_step}")

PART 16 COMPLETE
[PASS] Training sanity check completed.
Sanity training loss : 2.7121
Sanity global steps  : 2


# 17. Full Fine-Tuning

The sanity check has confirmed that the forward pass, dynamic audio feature
extraction, multilingual labels, loss computation, backpropagation, and
optimizer update can run successfully.

The full training run now uses the complete verified training set.

The validation set remains separate and is used for validation loss during
training. WER and CER are computed after training with language-aware
decoding.

In [90]:
# ============================================================
# 17.1 Install Memory-Efficient Optimizer
# ============================================================

!pip install -q -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.0 MB/s eta 0:00:00


In [91]:
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

In [92]:
# ============================================================
# 17.X Create Full Trainer
# ============================================================

from transformers import Seq2SeqTrainer

training_args.remove_unused_columns = False
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset["train"],
    eval_dataset=processed_dataset["validation"],
    data_collator=data_collator,
)

print("=" * 70)
print("FULL TRAINER READY")
print("=" * 70)

print("Train examples      :", len(processed_dataset["train"]))
print("Validation examples :", len(processed_dataset["validation"]))
print("Data collator       :", type(data_collator).__name__)
print("remove_unused_columns:", trainer.args.remove_unused_columns)

assert trainer.args.remove_unused_columns is False

print("\n[PASS] Full trainer is ready.")

FULL TRAINER READY
Train examples      : 2283
Validation examples : 508
Data collator       : DataCollatorSpeechSeq2SeqWithPadding
remove_unused_columns: False

[PASS] Full trainer is ready.


In [93]:
# ============================================================
# 17.2 Reload Paza Model for T4 Fine-Tuning
# ============================================================

import torch
from transformers import AutoModelForSpeechSeq2Seq

print("=" * 70)
print("LOADING PAZA MODEL")
print("=" * 70)

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

# Important for training
if hasattr(model, "config"):
    model.config.use_cache = False

print("Model loaded successfully.")
print("Dtype:", next(model.parameters()).dtype)

if torch.cuda.is_available():
    model = model.cuda()

    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Allocated GPU memory:",
        f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
    )

LOADING PAZA MODEL
Model loaded successfully.
Dtype: torch.float16
GPU: Tesla T4
Allocated GPU memory: 6.14 GB


In [94]:
# ============================================================
# 17.X Full Trainer DataLoader Test
# ============================================================

print("=" * 70)
print("TESTING FULL TRAINING DATALOADER")
print("=" * 70)

test_loader = trainer.get_train_dataloader()

first_batch = next(iter(test_loader))

print("\nBatch keys:")
print(first_batch.keys())

print("\nInput features:")
print(first_batch["input_features"].shape)

print("\nInput feature dtype:")
print(first_batch["input_features"].dtype)

print("\nLabels:")
print(first_batch["labels"].shape)

assert "input_features" in first_batch
assert "labels" in first_batch

print("\n[PASS] Full training DataLoader works.")
print("[PASS] audio_path was successfully preserved until collation.")

TESTING FULL TRAINING DATALOADER

Batch keys:
dict_keys(['input_features', 'labels'])

Input features:
torch.Size([1, 128, 3000])

Input feature dtype:
torch.float32

Labels:
torch.Size([1, 84])

[PASS] Full training DataLoader works.
[PASS] audio_path was successfully preserved until collation.


In [95]:
# ============================================================
# 17.1 Full Trainer
# ============================================================

from transformers import Seq2SeqTrainer

model.train()
model.config.use_cache = False

if torch.cuda.is_available():
    torch.cuda.empty_cache()

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset["train"],
    eval_dataset=processed_dataset["validation"],
    data_collator=data_collator,
)

print("[PASS] Full fine-tuning trainer created.")
print(f"Training examples   : {len(processed_dataset['train']):,}")
print(f"Validation examples : {len(processed_dataset['validation']):,}")

[PASS] Full fine-tuning trainer created.
Training examples   : 2,283
Validation examples : 508


In [96]:
# ============================================================
# 17.2 Full Training Verification
# ============================================================

assert len(processed_dataset["train"]) == 2283
assert len(processed_dataset["validation"]) == 508
assert training_args.per_device_train_batch_size == 1
assert training_args.gradient_accumulation_steps == 8
assert training_args.fp16 == torch.cuda.is_available()

print("[PASS] Full training dataset and configuration verified.")

[PASS] Full training dataset and configuration verified.


In [97]:
# ============================================================
# 17.X — Inspect Current Training Configuration
# ============================================================

print("=" * 70)
print("CURRENT TRAINING CONFIGURATION")
print("=" * 70)

print("training_args exists:", "training_args" in globals())

if "training_args" in globals():
    print("training_args type:", type(training_args))
    print("\nCurrent configuration:\n")
    print(training_args)

print("\nModel dtype:", next(model.parameters()).dtype)

print("\nDataset:")
print("Train      :", len(processed_dataset["train"]))
print("Validation :", len(processed_dataset["validation"]))

CURRENT TRAINING CONFIGURATION
training_args exists: True
training_args type: <class 'transformers.training_args_seq2seq.Seq2SeqTrainingArguments'>

Current configuration:

Seq2SeqTrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=2,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
dispatch_batches=None,
do_eval=True,
do_predict

In [98]:
# ============================================================
# 17.Z — FP32 Forward/Backward Compatibility Test
# ============================================================

import torch

print("=" * 70)
print("FP32 FORWARD / BACKWARD COMPATIBILITY TEST")
print("=" * 70)

# ------------------------------------------------------------
# 1. Ensure model is completely FP32
# ------------------------------------------------------------

model = model.float()
model.train()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Device:", device)
print("Model parameter dtype:", next(model.parameters()).dtype)

# ------------------------------------------------------------
# 2. Prepare one training example
# ------------------------------------------------------------

sample = processed_dataset["train"][0]
batch = data_collator([sample])

batch = {
    k: v.to(device) if torch.is_tensor(v) else v
    for k, v in batch.items()
}

print("\nBatch:")
for key, value in batch.items():
    if torch.is_tensor(value):
        print(
            f"{key:20s} "
            f"shape={tuple(value.shape)} "
            f"dtype={value.dtype}"
        )

# ------------------------------------------------------------
# 3. FP32 forward pass — NO autocast
# ------------------------------------------------------------

model.zero_grad(set_to_none=True)

print("\nRunning FP32 forward pass...")

outputs = model(**batch)
loss = outputs.loss

print("[PASS] Forward pass completed.")

print("Loss value :", loss.item())
print("Loss dtype :", loss.dtype)

# ------------------------------------------------------------
# 4. FP32 backward pass
# ------------------------------------------------------------

print("\nRunning FP32 backward pass...")

loss.backward()

print("[PASS] Backward pass completed.")

# ------------------------------------------------------------
# 5. Inspect gradients
# ------------------------------------------------------------

gradient_dtypes = {}
gradient_count = 0

for name, param in model.named_parameters():

    if param.grad is not None:

        gradient_count += 1

        dtype = str(param.grad.dtype)

        gradient_dtypes[dtype] = (
            gradient_dtypes.get(dtype, 0) + 1
        )

        if gradient_count <= 10:
            print(
                f"{name:60s} "
                f"param={param.dtype} "
                f"grad={param.grad.dtype}"
            )

print("\nGradient dtype summary:")

for dtype, count in gradient_dtypes.items():
    print(f"  {dtype}: {count} parameters")

print("\nParameters with gradients:", gradient_count)

print("\n" + "=" * 70)
print("FP32 TEST RESULT")
print("=" * 70)

if gradient_count > 0:
    print("[PASS] Paza Whisper supports FP32 forward + backward.")
else:
    print("[FAIL] No gradients were produced.")

print("=" * 70)

FP32 FORWARD / BACKWARD COMPATIBILITY TEST
Device: cuda
Model parameter dtype: torch.float32

Batch:
input_features       shape=(1, 128, 3000) dtype=torch.float32
labels               shape=(1, 14) dtype=torch.int64

Running FP32 forward pass...
[PASS] Forward pass completed.
Loss value : 3.9058430194854736
Loss dtype : torch.float32

Running FP32 backward pass...
[PASS] Backward pass completed.
model.encoder.conv1.weight                                   param=torch.float32 grad=torch.float32
model.encoder.conv1.bias                                     param=torch.float32 grad=torch.float32
model.encoder.conv2.weight                                   param=torch.float32 grad=torch.float32
model.encoder.conv2.bias                                     param=torch.float32 grad=torch.float32
model.encoder.layers.0.self_attn.k_proj.weight               param=torch.float32 grad=torch.float32
model.encoder.layers.0.self_attn.v_proj.weight               param=torch.float32 grad=torch.float32
m

# 18. Execute Full Fine-Tuning

This cell starts the actual fine-tuning run.

On a Tesla T4, the run may take a substantial amount of time because
the model is trained with a small per-device batch size and gradient
accumulation.

In [99]:
# ============================================================
# 18.0a — MUST RUN FIRST: Full GPU Reset
# ============================================================
# ⚠️  If you are re-running Part 18 after a previous attempt,
#     you MUST restart the runtime first:
#     Runtime → Restart session
#     Then run cells 1 through 17 before continuing here.
#
#     The OOM during from_pretrained() means a previous model
#     is still occupying ~13.7 GB of GPU memory. Python's
#     garbage collector cannot release GPU memory held by
#     PyTorch tensors that are still referenced inside the
#     trainer, optimizer, or dataloader workers.
# ============================================================

import os
import gc
import torch

# ✅ Set allocator config before ANY CUDA call
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("=" * 70)
print("GPU STATE BEFORE SETUP")
print("=" * 70)

if torch.cuda.is_available():
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    alloc_gb = torch.cuda.memory_allocated() / 1024 ** 3
    resv_gb  = torch.cuda.memory_reserved()  / 1024 ** 3
    free_gb  = total_gb - resv_gb

    print(f"GPU total     : {total_gb:.2f} GB")
    print(f"Allocated     : {alloc_gb:.2f} GB")
    print(f"Reserved      : {resv_gb:.2f} GB")
    print(f"Free          : {free_gb:.2f} GB")

    # ✅ Aggressively release every reference we can find
    for var in ["model", "trainer", "optimizer", "sanity_trainer"]:
        if var in globals():
            print(f"Deleting {var}...")
            del globals()[var]

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    alloc_after = torch.cuda.memory_allocated() / 1024 ** 3
    free_after  = (
        torch.cuda.get_device_properties(0).total_memory
        - torch.cuda.memory_reserved()
    ) / 1024 ** 3

    print(f"\nAfter cleanup:")
    print(f"Allocated     : {alloc_after:.2f} GB")
    print(f"Free          : {free_after:.2f} GB")

    if free_after < 5.0:
        print()
        print("=" * 70)
        print("⚠️  WARNING: Less than 5 GB free after cleanup.")
        print("    A previous model is still occupying GPU memory.")
        print("    You MUST restart the runtime before continuing:")
        print("    Runtime → Restart session → Re-run all cells.")
        print("=" * 70)
    else:
        print("\n[PASS] Enough free memory to proceed.")

GPU STATE BEFORE SETUP
GPU total     : 14.56 GB
Allocated     : 9.21 GB
Reserved      : 11.81 GB
Free          : 2.75 GB
Deleting model...
Deleting trainer...
Deleting sanity_trainer...

After cleanup:
Allocated     : 6.12 GB
Free          : 7.40 GB

[PASS] Enough free memory to proceed.


In [100]:
# ============================================================
# 18.0b — Load Model onto CPU First, Then Move to GPU
# ============================================================
#
# ✅ KEY FIX: Load to CPU first.
#
#    device_map="cuda:0" caused the OOM because it tried to
#    move each weight tensor to GPU one by one while the old
#    model's tensors were still fragmented in reserved memory.
#
#    Loading to CPU first keeps peak GPU usage near zero
#    during from_pretrained(). We then move the model to GPU
#    in one operation after the load completes, giving PyTorch
#    a chance to allocate a single contiguous block.
# ============================================================

import torch
from transformers import WhisperForConditionalGeneration

print("=" * 70)
print("LOADING PAZA MODEL TO CPU")
print("=" * 70)

# Load entirely to CPU — zero GPU impact during load
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    low_cpu_mem_usage=True,
    # ✅ No device_map here — CPU load first
)

print("Model loaded to CPU.")
print("Parameter dtype  :", next(model.parameters()).dtype)
print("Parameter device :", next(model.parameters()).device)

assert next(model.parameters()).dtype == torch.float32
assert next(model.parameters()).device.type == "cpu"

print("[PASS] Model loaded to CPU in FP32.")

LOADING PAZA MODEL TO CPU
Model loaded to CPU.
Parameter dtype  : torch.float32
Parameter device : cpu
[PASS] Model loaded to CPU in FP32.


In [101]:
# ============================================================
# 18.0c — Configure Model Then Move to GPU
# ============================================================

import torch

# ---------------------------------------------------------------
# 1. All configuration happens on CPU — zero GPU cost
# ---------------------------------------------------------------

model.config.use_cache = False

# ✅ Freeze encoder on CPU before moving to GPU.
#    This means frozen encoder weights are moved to GPU
#    but will never need gradient buffers or optimizer states,
#    saving ~6 GB of GPU memory during training.
for param in model.model.encoder.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())

print(f"Trainable parameters : {trainable:,}")
print(f"Total parameters     : {total:,}")
print(f"Trainable ratio      : {trainable / total * 100:.1f}%")

# ---------------------------------------------------------------
# 2. Enable gradient checkpointing on CPU
#    (configuration only — no GPU tensors involved)
# ---------------------------------------------------------------

model.enable_input_require_grads()
model.gradient_checkpointing_enable()

print("Encoder frozen         : YES")
print("Input require grads    : ENABLED")
print("Gradient checkpointing : ENABLED")

# ---------------------------------------------------------------
# 3. Move to GPU in one single operation
#
#    ✅ Moving after configuration means PyTorch allocates
#       one contiguous block for the whole model rather than
#       tensor-by-tensor allocations that fragment the heap.
# ---------------------------------------------------------------

if torch.cuda.is_available():
    torch.cuda.empty_cache()

    free_before = (
        torch.cuda.get_device_properties(0).total_memory
        - torch.cuda.memory_reserved()
    ) / 1024 ** 3

    print(f"\nFree GPU before model move : {free_before:.2f} GB")

    model = model.cuda()

    alloc_after = torch.cuda.memory_allocated() / 1024 ** 3
    free_after  = (
        torch.cuda.get_device_properties(0).total_memory
        - torch.cuda.memory_reserved()
    ) / 1024 ** 3

    print(f"GPU allocated after move   : {alloc_after:.2f} GB")
    print(f"GPU free after move        : {free_after:.2f} GB")

print("\nParameter dtype  :", next(model.parameters()).dtype)
print("Parameter device :", next(model.parameters()).device)

assert next(model.parameters()).device.type == "cuda"
assert model.config.use_cache is False

print("\n[PASS] Model on GPU.")
print("[PASS] Encoder frozen.")
print("[PASS] Gradient checkpointing enabled.")

Trainable parameters : 171,915,520
Total parameters     : 808,884,480
Trainable ratio      : 21.3%
Encoder frozen         : YES
Input require grads    : ENABLED
Gradient checkpointing : ENABLED

Free GPU before model move : 7.40 GB
GPU allocated after move   : 9.20 GB
GPU free after move        : 4.72 GB

Parameter dtype  : torch.float32
Parameter device : cuda:0

[PASS] Model on GPU.
[PASS] Encoder frozen.
[PASS] Gradient checkpointing enabled.


In [102]:
# ---------------------------------------------------------------
# 3. Disable KV cache (required for gradient checkpointing)
# ---------------------------------------------------------------

model.config.use_cache = False

print("use_cache:", model.config.use_cache)

# ---------------------------------------------------------------
# 4. Enable gradient checkpointing
#
#    ✅ FIX: Call enable_input_require_grads() before
#             gradient_checkpointing_enable().
#
#             Without this, the encoder's first module does not
#             receive a gradient-carrying input on the first
#             backward pass, causing encoder parameters to have
#             grad=None and silently producing a partial training
#             run where only the decoder is updated.
# ---------------------------------------------------------------

model.enable_input_require_grads()
model.gradient_checkpointing_enable()

print("Input require grads : ENABLED")
print("Gradient checkpointing : ENABLED")

use_cache: False
Input require grads : ENABLED
Gradient checkpointing : ENABLED


In [103]:
# ---------------------------------------------------------------
# 5. Training arguments
#
#    ✅ FIX: predict_with_generate=False during training.
#             Setting it True with no compute_metrics causes the
#             trainer to run full beam-search generation on every
#             eval step — wasting 2-4 GB of GPU memory and
#             doubling eval time — then silently discarding the
#             output because nothing consumes it.
#             Generation for WER/CER is handled separately in
#             Part 20 after training is complete.
#
#    ✅ FIX: load_best_model_at_end=True so the saved model is
#             the checkpoint with lowest validation loss, not
#             simply the last checkpoint.
# ---------------------------------------------------------------

training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINT_ROOT),

    # ----------------------------------------------------------
    # T4 memory budget
    # ----------------------------------------------------------
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    # ----------------------------------------------------------
    # Optimization
    # ----------------------------------------------------------
    learning_rate=1e-5,
    num_train_epochs=3,
    warmup_ratio=0.05,

    # ----------------------------------------------------------
    # Logging
    # ----------------------------------------------------------
    logging_strategy="steps",
    logging_steps=10,

    # ----------------------------------------------------------
    # Evaluation and checkpointing
    # ----------------------------------------------------------
    eval_strategy="steps",
    eval_steps=100,

    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    # ✅ FIX: Keep the best checkpoint, not just the last one.
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # ----------------------------------------------------------
    # Mixed precision
    # ----------------------------------------------------------
    fp16=True,
    bf16=False,

    # ----------------------------------------------------------
    # Memory-efficient optimizer
    # ----------------------------------------------------------
    optim="adamw_bnb_8bit",

    # ----------------------------------------------------------
    # Data loading
    # ----------------------------------------------------------
    dataloader_num_workers=0,

    # Required by the custom audio collator
    remove_unused_columns=False,

    # ✅ FIX: False during training — no compute_metrics is
    #         defined, so generation here wastes GPU memory.
    predict_with_generate=False,

    # ----------------------------------------------------------
    # Reporting and reproducibility
    # ----------------------------------------------------------
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

print("=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

print(f"Output dir             : {training_args.output_dir}")
print(f"Train batch size       : {training_args.per_device_train_batch_size}")
print(f"Gradient accumulation  : {training_args.gradient_accumulation_steps}")
print(f"Effective batch size   : "
      f"{training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Learning rate          : {training_args.learning_rate}")
print(f"Epochs                 : {training_args.num_train_epochs}")
print(f"FP16                   : {training_args.fp16}")
print(f"Optimizer              : {training_args.optim}")
print(f"Gradient checkpointing : {training_args.gradient_checkpointing}")
print(f"predict_with_generate  : {training_args.predict_with_generate}")
print(f"load_best_model_at_end : {training_args.load_best_model_at_end}")
print(f"remove_unused_columns  : {training_args.remove_unused_columns}")

# Assertions
assert training_args.fp16 is True
assert training_args.bf16 is False
assert training_args.optim == "adamw_bnb_8bit"
assert training_args.remove_unused_columns is False
assert training_args.predict_with_generate is False
assert training_args.load_best_model_at_end is True
assert model.config.use_cache is False

print("\n[PASS] Training arguments verified.")

TRAINING CONFIGURATION
Output dir             : /content/multilingual_kenyan_asr/outputs/checkpoints
Train batch size       : 1
Gradient accumulation  : 8
Effective batch size   : 8
Learning rate          : 1e-05
Epochs                 : 3
FP16                   : True
Optimizer              : OptimizerNames.ADAMW_BNB
Gradient checkpointing : False
predict_with_generate  : False
load_best_model_at_end : True
remove_unused_columns  : False

[PASS] Training arguments verified.


In [104]:
# ============================================================
# 18.0e — Create Trainer
# ============================================================

from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset["train"],
    eval_dataset=processed_dataset["validation"],
    data_collator=data_collator,
)

print("=" * 70)
print("PRE-TRAINING VERIFICATION")
print("=" * 70)

print("Model dtype            :", next(model.parameters()).dtype)
print("Model device           :", next(model.parameters()).device)
print("FP16                   :", training_args.fp16)
print("Optimizer              :", training_args.optim)
print("Encoder frozen         : YES")
print("Input require grads    : ENABLED")
print("Gradient checkpointing : ENABLED")
print("use_cache              :", model.config.use_cache)
print("Train examples         :", len(processed_dataset["train"]))
print("Validation examples    :", len(processed_dataset["validation"]))

if torch.cuda.is_available():
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    alloc_gb = torch.cuda.memory_allocated() / 1024 ** 3
    free_gb  = total_gb - torch.cuda.memory_reserved() / 1024 ** 3
    print(f"GPU total  : {total_gb:.2f} GB")
    print(f"GPU alloc  : {alloc_gb:.2f} GB")
    print(f"GPU free   : {free_gb:.2f} GB")

    if free_gb < 3.0:
        print()
        print("⚠️  WARNING: Less than 3 GB free before training.")
        print("    Consider restarting the runtime and re-running.")

print()
print("[PASS] Model on CUDA.")
print("[PASS] FP16 AMP enabled.")
print("[PASS] Encoder frozen.")
print("[PASS] Gradient checkpointing enabled.")
print("[PASS] Trainer created successfully.")
print("[PASS] Ready for Cell 18.1.")

PRE-TRAINING VERIFICATION
Model dtype            : torch.float32
Model device           : cuda:0
FP16                   : True
Optimizer              : OptimizerNames.ADAMW_BNB
Encoder frozen         : YES
Input require grads    : ENABLED
Gradient checkpointing : ENABLED
use_cache              : False
Train examples         : 2283
Validation examples    : 508
GPU total  : 14.56 GB
GPU alloc  : 9.20 GB
GPU free   : 4.72 GB

[PASS] Model on CUDA.
[PASS] FP16 AMP enabled.
[PASS] Encoder frozen.
[PASS] Gradient checkpointing enabled.
[PASS] Trainer created successfully.
[PASS] Ready for Cell 18.1.


In [ ]:
# ============================================================
# 18.1 Run Full Fine-Tuning
# ============================================================

print("=" * 70)
print("STARTING FULL MULTILINGUAL PAZA FINE-TUNING")
print("=" * 70)

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    free_gb = (
        torch.cuda.get_device_properties(0).total_memory
        - torch.cuda.memory_reserved()
    ) / 1024 ** 3
    print(f"Free GPU memory before training : {free_gb:.2f} GB")

try:
    train_result = trainer.train()

except RuntimeError as e:
    if "out of memory" in str(e).lower():
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print("\n" + "=" * 70)
        print("OUT OF MEMORY — TRAINING FAILED")
        print("=" * 70)
        print("Remaining options:")
        print("  1. Runtime → Restart session → Re-run all cells")
        print("  2. Reduce gradient_accumulation_steps to 2")
        print("  3. Reduce num_train_epochs to 1")
        print("  4. Confirm PYTORCH_CUDA_ALLOC_CONF is set")
    raise

print("\n" + "=" * 70)
print("FULL FINE-TUNING FINISHED")
print("=" * 70)

print(f"Training loss  : {train_result.training_loss:.4f}")
print(f"Global steps   : {train_result.global_step}")
print(
    f"Runtime (s)    : "
    f"{train_result.metrics.get('train_runtime', 'N/A'):.1f}"
)
print(
    f"Samples/sec    : "
    f"{train_result.metrics.get('train_samples_per_second', 'N/A'):.2f}"
)

if torch.cuda.is_available():
    peak_gb = torch.cuda.max_memory_allocated() / 1024 ** 3
    print(f"Peak GPU memory : {peak_gb:.2f} GB")

STARTING FULL MULTILINGUAL PAZA FINE-TUNING
Free GPU memory before training : 4.72 GB


/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss,Validation Loss
100,0.432900,0.415435
200,0.434900,0.378309


/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss,Validation Loss
100,0.432900,0.415435
200,0.434900,0.378309
300,0.196000,0.363698
400,0.216200,0.362651


/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


In [ ]:
# ============================================================
# 18.2 Save Training State
# ============================================================

trainer.save_state()

print("[PASS] Trainer state saved.")
print(f"Checkpoint root : {CHECKPOINT_ROOT}")

# 19. Final Model Export

The final fine-tuned model, processor, tokenizer, and configuration are
saved together so that the model can be reloaded independently of the
training notebook.

In [ ]:
# ============================================================
# 19. FINAL FINE-TUNED MODEL — SAVE TO GOOGLE DRIVE
# ============================================================

from pathlib import Path
import shutil
import torch

print("=" * 70)
print("SAVING FINAL FINE-TUNED PAZA MODEL")
print("=" * 70)

# ------------------------------------------------------------
# ✅ FIX 1: Define FINAL_MODEL_ROOT as a Google Drive path,
#            not a local /content/ path.
#
#            Local /content/ paths are wiped when the Colab
#            session ends. The Drive path persists permanently.
#            trainer.save_model() writes directly to Drive so
#            no manual copy step is needed.
# ------------------------------------------------------------

FINAL_MODEL_ROOT = Path(
    "/content/drive/MyDrive"
    "/multilingual_kenyan_asr"
    "/outputs"
    "/final_model"
)

FINAL_MODEL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"\nSaving to Google Drive:")
print(FINAL_MODEL_ROOT)

# ------------------------------------------------------------
# ✅ FIX 2: Verify the best checkpoint was loaded before saving.
#
#    With load_best_model_at_end=True the trainer reloads the
#    best checkpoint at the end of train(). If that reload
#    silently failed (e.g. OOM during final eval), model still
#    holds the last step's weights, not the best ones.
#    We verify by checking trainer.state.best_model_checkpoint.
# ------------------------------------------------------------

best_ckpt = getattr(trainer.state, "best_model_checkpoint", None)

if best_ckpt is not None:
    print(f"\nBest checkpoint : {best_ckpt}")
    print(
        f"Best eval loss  : "
        f"{trainer.state.best_metric:.4f}"
    )
else:
    print(
        "\n⚠️  WARNING: best_model_checkpoint is None. "
        "The model being saved may not be the best checkpoint."
    )

# ------------------------------------------------------------
# Prepare model for inference
# ------------------------------------------------------------

model.config.use_cache = True
model.eval()

# ------------------------------------------------------------
# Save model weights and configuration directly to Drive
# ------------------------------------------------------------

trainer.save_model(
    str(FINAL_MODEL_ROOT)
)

# ------------------------------------------------------------
# Save processor / tokenizer / feature extractor
# ------------------------------------------------------------

processor.save_pretrained(
    str(FINAL_MODEL_ROOT)
)

# ------------------------------------------------------------
# ✅ FIX 3: Save trainer state to CHECKPOINT_ROOT, not to
#            FINAL_MODEL_ROOT.
#
#            Trainer state contains optimizer, scheduler, and
#            RNG state — it belongs with the training checkpoints,
#            not in the final exported model directory.
# ------------------------------------------------------------

trainer.save_state()

print("\n" + "=" * 70)
print("FINAL MODEL SAVED TO GOOGLE DRIVE")
print("=" * 70)

# ------------------------------------------------------------
# List saved files
# ------------------------------------------------------------

print("\nSaved files:")

for path in sorted(FINAL_MODEL_ROOT.iterdir()):

    if path.is_file():

        size_mb = path.stat().st_size / (1024 ** 2)

        print(
            f"  {path.name:<40} "
            f"{size_mb:>10.2f} MB"
        )

print(f"\nGoogle Drive path : {FINAL_MODEL_ROOT}")

In [ ]:
# ============================================================
# 19.1 VERIFY FINAL MODEL FILES
# ============================================================

print("=" * 70)
print("VERIFYING FINAL MODEL")
print("=" * 70)

required_files = [
    "config.json",
    "generation_config.json",
    "preprocessor_config.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "model.safetensors",
]

all_present = True

for filename in required_files:

    path = FINAL_MODEL_ROOT / filename

    if path.exists():

        size_mb = path.stat().st_size / (1024 ** 2)

        print(
            f"[PASS] {filename:<35} "
            f"{size_mb:>10.2f} MB"
        )

    else:

        print(
            f"[FAIL] {filename:<35} "
            f"MISSING"
        )

        all_present = False

if not all_present:
    raise FileNotFoundError(
        "\nFINAL MODEL IS INCOMPLETE. "
        "Do not disconnect Colab yet."
    )

# ✅ Verify safetensors is not suspiciously small
#    (a near-zero file means the save silently failed)
safetensors_path = FINAL_MODEL_ROOT / "model.safetensors"
safetensors_gb   = safetensors_path.stat().st_size / (1024 ** 3)

print(f"\nmodel.safetensors size : {safetensors_gb:.3f} GB")

assert safetensors_gb > 0.5, (
    f"model.safetensors is suspiciously small "
    f"({safetensors_gb:.3f} GB). "
    f"The save may have failed silently."
)

print("\n" + "=" * 70)
print("FINAL MODEL VERIFICATION PASSED")
print("=" * 70)
print(f"\nModel saved to Google Drive:")
print(f"  {FINAL_MODEL_ROOT}")

In [ ]:
# ============================================================
# 19.2 RELOAD FINAL MODEL FOR EVALUATION
# ============================================================
# ✅ FIX 4: This cell was missing entirely.
#            Part 20.1 uses `reloaded_model` which was never
#            defined — causing NameError on every evaluation run.
#            Load the saved model back from Drive here.
# ============================================================

import gc
import torch
from transformers import WhisperForConditionalGeneration

print("=" * 70)
print("RELOADING FINAL MODEL FROM GOOGLE DRIVE")
print("=" * 70)

# Free the training model from GPU before loading
# the reloaded copy — avoids OOM with two models on GPU
if "model" in globals():
    model.cpu()
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

EVAL_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

reloaded_model = WhisperForConditionalGeneration.from_pretrained(
    str(FINAL_MODEL_ROOT),
    low_cpu_mem_usage=True,
)

reloaded_model = reloaded_model.to(EVAL_DEVICE)
reloaded_model.eval()

# Re-enable KV cache for inference — faster generation
reloaded_model.config.use_cache = True

print(f"Reloaded from    : {FINAL_MODEL_ROOT}")
print(f"Evaluation device: {EVAL_DEVICE}")
print(f"Parameter dtype  : {next(reloaded_model.parameters()).dtype}")
print(f"Parameter device : {next(reloaded_model.parameters()).device}")
print(f"use_cache        : {reloaded_model.config.use_cache}")

if torch.cuda.is_available():
    alloc_gb = torch.cuda.memory_allocated() / 1024 ** 3
    print(f"GPU allocated    : {alloc_gb:.2f} GB")

print("\n[PASS] Reloaded model ready for evaluation.")

# 20. Validation Evaluation: WER and CER

WER and CER are computed on the held-out validation set.

Because the experiment is multilingual and Paza uses language-specific
decoder tokens, each example is decoded with its own language token.

The metrics are therefore calculated from actual generated transcripts,
rather than from training loss.

In [ ]:

# ============================================================
# RELOAD THE FINE-TUNED MODEL FROM GOOGLE DRIVE
# ============================================================

import torch
from transformers import (
    WhisperForConditionalGeneration,
    AutoProcessor,
)
from pathlib import Path

FINAL_MODEL_ROOT = Path(
    "/content/drive/MyDrive/multilingual_kenyan_asr/"
    "outputs/final_model"
)

assert FINAL_MODEL_ROOT.exists(), (
    f"Model directory not found: {FINAL_MODEL_ROOT}"
)

assert (FINAL_MODEL_ROOT / "model.safetensors").is_file(), (
    "model.safetensors is missing from Google Drive"
)

# Reload the saved processor and fine-tuned weights
processor = AutoProcessor.from_pretrained(
    str(FINAL_MODEL_ROOT)
)

reloaded_model = WhisperForConditionalGeneration.from_pretrained(
    str(FINAL_MODEL_ROOT),
    low_cpu_mem_usage=True,
)

# Restore the model's inference configuration
reloaded_model.config.use_cache = True

EVAL_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

reloaded_model = reloaded_model.to(EVAL_DEVICE)
reloaded_model.eval()

# Use the tokenizer and feature extractor from the saved processor
tokenizer = processor.tokenizer
feature_extractor = processor.feature_extractor

print("Model reloaded successfully.")
print("Device:", EVAL_DEVICE)
print("Model directory:", FINAL_MODEL_ROOT)


In [ ]:
# ============================================================
# 20.1 Evaluation Dependencies
# ============================================================
# ✅ reloaded_model is now defined in Cell 19.2 above.
#    This cell no longer needs to call .to() or .eval() —
#    both are already done during reload.
# ============================================================

import numpy as np
import pandas as pd
from jiwer import wer, cer

print("Evaluation device :", EVAL_DEVICE)
print("Model eval mode   :", not reloaded_model.training)
print("[PASS] Evaluation dependencies ready.")

In [ ]:
# ============================================================
# 20.2 Language-Aware Generation
# ============================================================

def build_forced_decoder_ids(language_code):
    # Paza decoder prefix:
    # position 1 = language token
    # position 2 = transcribe token
    # position 3 = no-timestamps token
    return [
        [1, PAZA_LANGUAGE_TOKEN_IDS[language_code]],
        [2, TRANSCRIBE_TOKEN_ID],
        [3, NO_TIMESTAMPS_TOKEN_ID],
    ]


def transcribe_with_language(audio_path, language_name):
    audio = prepare_audio(audio_path)

    inputs = feature_extractor(
        audio["array"],
        sampling_rate=TARGET_SAMPLE_RATE,
        return_tensors="pt",
    )

    input_features = inputs["input_features"].to(EVAL_DEVICE)

    language_code = LANGUAGE_CODES[language_name]
    forced_decoder_ids = build_forced_decoder_ids(language_code)

    with torch.no_grad():
        generated_ids = reloaded_model.generate(
            input_features,
            forced_decoder_ids=forced_decoder_ids,
            max_new_tokens=128,
            do_sample=False,
        )

    prediction = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True,
    )[0].strip()

    return prediction

In [ ]:
# ============================================================
# 20.3 Validation WER/CER Evaluation
# ============================================================

evaluation_records = []
validation_frame = validation_df.reset_index(drop=True)

for idx, row in validation_frame.iterrows():

    reference = str(row["text"]).strip()
    prediction = transcribe_with_language(
        row["audio_path"],
        row["language"],
    )

    evaluation_records.append({
        "index": idx,
        "language": row["language"],
        "reference": reference,
        "prediction": prediction,
    })

    if (idx + 1) % 25 == 0:
        print(
            f"Evaluated {idx + 1}/{len(validation_frame)} "
            "validation samples"
        )

evaluation_df = pd.DataFrame(evaluation_records)

overall_wer = wer(
    evaluation_df["reference"].tolist(),
    evaluation_df["prediction"].tolist(),
)

overall_cer = cer(
    evaluation_df["reference"].tolist(),
    evaluation_df["prediction"].tolist(),
)

print("=" * 70)
print("OVERALL VALIDATION RESULTS")
print("=" * 70)
print(f"Samples : {len(evaluation_df):,}")
print(f"WER     : {overall_wer:.4f} ({overall_wer * 100:.2f}%)")
print(f"CER     : {overall_cer:.4f} ({overall_cer * 100:.2f}%)")

In [ ]:
# ============================================================
# 20.4 Per-Language WER/CER
# ============================================================

language_results = []

for language in SANITY_LANGUAGES:

    subset = evaluation_df[
        evaluation_df["language"] == language
    ]

    if len(subset) == 0:
        continue

    language_wer = wer(
        subset["reference"].tolist(),
        subset["prediction"].tolist(),
    )

    language_cer = cer(
        subset["reference"].tolist(),
        subset["prediction"].tolist(),
    )

    language_results.append({
        "language": language,
        "samples": len(subset),
        "WER": language_wer,
        "WER_percent": language_wer * 100,
        "CER": language_cer,
        "CER_percent": language_cer * 100,
    })

language_results_df = pd.DataFrame(language_results)

print("=" * 70)
print("PER-LANGUAGE VALIDATION RESULTS")
print("=" * 70)
print(language_results_df.to_string(index=False))

# 21. Qualitative Error Inspection

A small set of reference/prediction pairs is displayed to identify common
recognition problems such as substitutions, deletions, insertions,
truncation, and language-specific spelling differences.

In [ ]:
# ============================================================
# 21.1 Display Sample Predictions
# ============================================================

pd.set_option("display.max_colwidth", 200)

sample_predictions = evaluation_df[
    ["language", "reference", "prediction"]
].head(20)

display(sample_predictions)

# 22. Final Inference Test

The final exported model is tested on one validation example from each
supported language.

This verifies that the saved model can be used independently for
Swahili, Kikuyu, and Kalenjin inference.

In [ ]:
# ============================================================
# 22.1 One Example Per Language
# ============================================================

print("=" * 70)
print("FINAL MULTILINGUAL INFERENCE TEST")
print("=" * 70)

for language in SANITY_LANGUAGES:

    row = validation_df[
        validation_df["language"] == language
    ].iloc[0]

    prediction = transcribe_with_language(
        row["audio_path"],
        language,
    )

    print(f"\nLanguage   : {language}")
    print(f"Reference  : {row['text']}")
    print(f"Prediction : {prediction}")

print("\n[PASS] Final inference test completed for all languages.")

# 23. Experiment Summary

The notebook has now completed the intended ASR fine-tuning workflow:

1. Environment and dependency verification.
2. Google Drive dataset access and integrity verification.
3. Paza Whisper model and processor loading.
4. Audio preprocessing and 16 kHz normalization.
5. Multilingual decoder-label construction.
6. Label-length and dataset verification.
7. Memory-efficient dynamic feature extraction.
8. Data-collator verification.
9. Training configuration.
10. Training sanity check.
11. Full multilingual fine-tuning.
12. Final model export and reload verification.
13. Validation WER/CER evaluation.
14. Per-language and qualitative error analysis.
15. Final multilingual inference testing.

The notebook does not invent final performance values. WER, CER, loss,
and memory results are populated only when the corresponding cells are
executed successfully.

In [ ]:
# ============================================================
# 23.1 Final Experiment Record
# ============================================================

summary = {
    "model": MODEL_NAME,
    "languages": SANITY_LANGUAGES,
    "train_samples": len(processed_dataset["train"]),
    "validation_samples": len(processed_dataset["validation"]),
    "sanity_steps": 2,
    "sanity_training_loss": float(sanity_result.training_loss),
    "full_training_loss": float(train_result.training_loss),
    "overall_validation_WER": float(overall_wer),
    "overall_validation_CER": float(overall_cer),
    "final_model_directory": str(FINAL_MODEL_ROOT),
}

summary_file = OUTPUT_ROOT / "experiment_summary.json"

with open(summary_file, "w") as f:
    json.dump(summary, f, indent=4)

print("=" * 70)
print("EXPERIMENT SUMMARY SAVED")
print("=" * 70)

for key, value in summary.items():
    print(f"{key:<28}: {value}")

print(f"\nSaved to: {summary_file}")
print("[PASS] MULTILINGUAL PAZA ASR WORKFLOW COMPLETE.")

In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.endswith((".bin", ".safetensors", ".json")):
            print(os.path.join(root, file))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
import os

source = "/content/multilingual_kenyan_asr/outputs/final_model"
destination = "/content/drive/MyDrive/multilingual_kenyan_asr/outputs/final_model"

os.makedirs(os.path.dirname(destination), exist_ok=True)

shutil.copytree(source, destination, dirs_exist_ok=True)

print("Model backed up to:")
print(destination)

In [ ]:
!ls -lh /content/drive/MyDrive/multilingual_kenyan_asr/outputs/final_model

In [ ]:
!zip -r /content/final_model.zip \
/content/drive/MyDrive/multilingual_kenyan_asr/outputs/final_model

In [ ]:
!zip -r /content/drive/MyDrive/final_model.zip \
/content/drive/MyDrive/multilingual_kenyan_asr/outputs/final_model

In [ ]:
# ============================================================
# DIAGNOSE — What is actually in the final model folder?
# ============================================================

import os
from pathlib import Path

FINAL_MODEL_ROOT = Path(
    "/content/drive/MyDrive"
    "/multilingual_kenyan_asr"
    "/outputs"
    "/final_model"
)

print("=" * 70)
print("CONTENTS OF FINAL MODEL FOLDER")
print("=" * 70)

if not FINAL_MODEL_ROOT.exists():
    print("❌ Directory does not exist.")
else:
    all_files = list(FINAL_MODEL_ROOT.rglob("*"))

    if not all_files:
        print("❌ Directory exists but is EMPTY.")
    else:
        for path in sorted(all_files):
            if path.is_file():
                size_mb = path.stat().st_size / (1024 ** 2)
                print(f"  {path.name:<45} {size_mb:>10.2f} MB")

# Also check the local /content/ path in case save went there
LOCAL_PATH = Path(
    "/content/multilingual_kenyan_asr/outputs/final_model"
)

print("\n" + "=" * 70)
print("CONTENTS OF LOCAL /content/ FOLDER")
print("=" * 70)

if not LOCAL_PATH.exists():
    print("Directory does not exist.")
else:
    for path in sorted(LOCAL_PATH.rglob("*")):
        if path.is_file():
            size_mb = path.stat().st_size / (1024 ** 2)
            print(f"  {path.name:<45} {size_mb:>10.2f} MB")

# Also check all checkpoints
CHECKPOINT_ROOT = Path(
    "/content/multilingual_kenyan_asr/outputs/checkpoints"
)

print("\n" + "=" * 70)
print("CHECKPOINTS AVAILABLE")
print("=" * 70)

if not CHECKPOINT_ROOT.exists():
    print("No checkpoints directory found.")
else:
    for path in sorted(CHECKPOINT_ROOT.rglob("*.safetensors")):
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"  {path}  {size_mb:.1f} MB")

    for path in sorted(CHECKPOINT_ROOT.rglob("*.bin")):
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"  {path}  {size_mb:.1f} MB")

In [ ]:
# ============================================================
# SAVE FINAL MODEL FROM BEST CHECKPOINT TO DRIVE
# ============================================================

import gc
import torch
from pathlib import Path
from transformers import WhisperForConditionalGeneration, AutoProcessor

# ── Paths ─────────────────────────────────────────────────────
# checkpoint-855 is the last checkpoint — use it as the final
# model. If trainer.state.best_model_checkpoint points to
# checkpoint-800, use that one instead (lower eval loss).

BEST_CHECKPOINT = Path(
    "/content/multilingual_kenyan_asr"
    "/outputs/checkpoints/checkpoint-855"
)

DRIVE_PATH = Path(
    "/content/drive/MyDrive"
    "/multilingual_kenyan_asr"
    "/outputs"
    "/final_model"
)

# ── Check which checkpoint is actually best ───────────────────
print("=" * 70)
print("CHECKPOINT SELECTION")
print("=" * 70)

if "trainer" in globals():
    best_ckpt = getattr(trainer.state, "best_model_checkpoint", None)
    best_metric = getattr(trainer.state, "best_metric", None)

    if best_ckpt is not None:
        print(f"Trainer best checkpoint : {best_ckpt}")
        print(f"Trainer best eval loss  : {best_metric}")
        BEST_CHECKPOINT = Path(best_ckpt)
    else:
        print("trainer.state.best_model_checkpoint is None.")
        print(f"Using checkpoint-855 (last checkpoint).")
else:
    print("trainer not in scope — using checkpoint-855.")

print(f"\nLoading from : {BEST_CHECKPOINT}")
assert BEST_CHECKPOINT.exists(), f"Checkpoint not found: {BEST_CHECKPOINT}"
assert (BEST_CHECKPOINT / "model.safetensors").exists(), (
    f"model.safetensors missing from {BEST_CHECKPOINT}"
)

safetensors_mb = (
    BEST_CHECKPOINT / "model.safetensors"
).stat().st_size / (1024 ** 2)

print(f"model.safetensors size : {safetensors_mb:.1f} MB  ✓")

In [ ]:
from google.colab import files
files.download("/content/final_model.zip")

In [ ]:
# ============================================================
# ZIP AND DOWNLOAD MODEL FROM DRIVE
# ============================================================

import shutil
import os
from google.colab import files
from pathlib import Path

print("=" * 70)
print("ZIPPING FINAL MODEL FOR DOWNLOAD")
print("=" * 70)

# ── Zip the Drive folder to a local file ─────────────────────
ZIP_PATH = Path("/content/paza_whisper_kenyan.zip")

shutil.make_archive(
    base_name=str(ZIP_PATH.with_suffix("")),  # path without .zip
    format="zip",
    root_dir=str(FINAL_MODEL_ROOT.parent),    # parent directory
    base_dir=FINAL_MODEL_ROOT.name,           # folder to zip
)

size_mb = ZIP_PATH.stat().st_size / (1024 ** 2)
print(f"Zip created : {ZIP_PATH}")
print(f"Zip size    : {size_mb:.1f} MB")

# ── Download to your computer ─────────────────────────────────
print("\nStarting download...")
files.download(str(ZIP_PATH))
print("[PASS] Download started.")

In [ ]:
# ============================================================
# SAVE FINAL MODEL FROM BEST CHECKPOINT TO DRIVE
# ============================================================

import gc
import torch
from pathlib import Path
from transformers import WhisperForConditionalGeneration, AutoProcessor

# ── Paths ─────────────────────────────────────────────────────
# checkpoint-855 is the last checkpoint — use it as the final
# model. If trainer.state.best_model_checkpoint points to
# checkpoint-800, use that one instead (lower eval loss).

BEST_CHECKPOINT = Path(
    "/content/multilingual_kenyan_asr"
    "/outputs/checkpoints/checkpoint-855"
)

DRIVE_PATH = Path(
    "/content/drive/MyDrive"
    "/multilingual_kenyan_asr"
    "/outputs"
    "/final_model"
)

# ── Check which checkpoint is actually best ───────────────────
print("=" * 70)
print("CHECKPOINT SELECTION")
print("=" * 70)

if "trainer" in globals():
    best_ckpt = getattr(trainer.state, "best_model_checkpoint", None)
    best_metric = getattr(trainer.state, "best_metric", None)

    if best_ckpt is not None:
        print(f"Trainer best checkpoint : {best_ckpt}")
        print(f"Trainer best eval loss  : {best_metric}")
        BEST_CHECKPOINT = Path(best_ckpt)
    else:
        print("trainer.state.best_model_checkpoint is None.")
        print(f"Using checkpoint-855 (last checkpoint).")
else:
    print("trainer not in scope — using checkpoint-855.")

print(f"\nLoading from : {BEST_CHECKPOINT}")
assert BEST_CHECKPOINT.exists(), f"Checkpoint not found: {BEST_CHECKPOINT}"
assert (BEST_CHECKPOINT / "model.safetensors").exists(), (
    f"model.safetensors missing from {BEST_CHECKPOINT}"
)

safetensors_mb = (
    BEST_CHECKPOINT / "model.safetensors"
).stat().st_size / (1024 ** 2)

print(f"model.safetensors size : {safetensors_mb:.1f} MB  ✓")

In [ ]:
# ── Load from checkpoint ──────────────────────────────────────
print("=" * 70)
print("LOADING FROM CHECKPOINT")
print("=" * 70)

# Free GPU before loading — avoids OOM with two model copies
for var in ["model", "reloaded_model", "model_to_save"]:
    if var in globals():
        globals()[var].cpu()
        del globals()[var]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model_to_save = WhisperForConditionalGeneration.from_pretrained(
    str(BEST_CHECKPOINT),
    low_cpu_mem_usage=True,
)

model_to_save.config.use_cache = True
model_to_save.eval()

print(f"Loaded successfully.")
print(f"Parameters : {sum(p.numel() for p in model_to_save.parameters()):,}")

In [ ]:
# ── Save to Drive with safetensors ───────────────────────────
print("=" * 70)
print("SAVING FINAL MODEL TO GOOGLE DRIVE")
print("=" * 70)

DRIVE_PATH.mkdir(parents=True, exist_ok=True)

# ✅ safe_serialization=True forces model.safetensors format
model_to_save.save_pretrained(
    str(DRIVE_PATH),
    safe_serialization=True,
)

processor.save_pretrained(str(DRIVE_PATH))

print("\n" + "=" * 70)
print("SAVED FILES")
print("=" * 70)

all_present = True
required = [
    "model.safetensors",
    "config.json",
    "generation_config.json",
    "preprocessor_config.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
]

for path in sorted(DRIVE_PATH.iterdir()):
    if path.is_file():
        size_mb = path.stat().st_size / (1024 ** 2)
        tag = "✓" if path.name in required else " "
        print(f"  {tag} {path.name:<45} {size_mb:>10.2f} MB")

for filename in required:
    if not (DRIVE_PATH / filename).exists():
        print(f"\n[FAIL] {filename} is MISSING")
        all_present = False

assert all_present, "Final model is incomplete — do not disconnect Colab."

safetensors_gb = (
    DRIVE_PATH / "model.safetensors"
).stat().st_size / (1024 ** 3)

assert safetensors_gb > 0.5, (
    f"model.safetensors is too small ({safetensors_gb:.3f} GB) "
    f"— save may have failed."
)

print(f"\n[PASS] model.safetensors : {safetensors_gb:.3f} GB")
print(f"[PASS] Final model saved to Google Drive.")
print(f"\n       {DRIVE_PATH}")

In [ ]:
# ── Clean up ─────────────────────────────────────────────────
del model_to_save
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("[PASS] GPU memory cleared.")

In [ ]:
# ============================================================
# CONFIRM MODEL IS SAVED CORRECTLY ON DRIVE
# ============================================================

from pathlib import Path

DRIVE_PATH = Path(
    "/content/drive/MyDrive"
    "/multilingual_kenyan_asr"
    "/outputs"
    "/final_model"
)

print("=" * 70)
print("FINAL MODEL — DRIVE VERIFICATION")
print("=" * 70)

required_files = [
    "model.safetensors",
    "config.json",
    "generation_config.json",
    "preprocessor_config.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
]

all_present  = True
total_size   = 0

print(f"\nLocation : {DRIVE_PATH}\n")

for path in sorted(DRIVE_PATH.iterdir()):
    if path.is_file():
        size_mb    = path.stat().st_size / (1024 ** 2)
        total_size += size_mb
        tag        = "✓" if path.name in required_files else " "
        print(f"  {tag}  {path.name:<45} {size_mb:>10.2f} MB")

print(f"\n  {'Total':<47} {total_size:>10.2f} MB")

print("\n" + "=" * 70)
print("REQUIRED FILES CHECK")
print("=" * 70)

for filename in required_files:
    path = DRIVE_PATH / filename
    if path.exists():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"  [PASS]  {filename:<40} {size_mb:>10.2f} MB")
    else:
        print(f"  [FAIL]  {filename:<40}      MISSING ❌")
        all_present = False

# ── Safetensors size sanity check ─────────────────────────────
if (DRIVE_PATH / "model.safetensors").exists():
    safetensors_gb = (
        DRIVE_PATH / "model.safetensors"
    ).stat().st_size / (1024 ** 3)

    print(f"\n  model.safetensors : {safetensors_gb:.3f} GB")

    if safetensors_gb > 0.5:
        print("  [PASS]  Weights file is a valid size.")
    else:
        print("  [FAIL]  Weights file is too small — save failed. ❌")
        all_present = False

print("\n" + "=" * 70)

if all_present:
    print("✅  MODEL SAVE VERIFIED — ALL FILES PRESENT")
else:
    print("❌  MODEL SAVE INCOMPLETE — SEE FAILS ABOVE")

print("=" * 70)

In [ ]:
# ============================================================
# ZIP AND DOWNLOAD FINAL MODEL
# ============================================================

import shutil
from pathlib import Path
from google.colab import files

DRIVE_PATH = Path(
    "/content/drive/MyDrive"
    "/multilingual_kenyan_asr"
    "/outputs"
    "/final_model"
)

ZIP_PATH = Path("/content/paza_whisper_kenyan_final.zip")

# ── Remove old zip if it exists ───────────────────────────────
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
    print(f"Removed old zip.")

# ── Create zip ────────────────────────────────────────────────
print("=" * 70)
print("ZIPPING FINAL MODEL")
print("=" * 70)

print("Zipping... (this may take 1-2 minutes for a 3 GB model)")

shutil.make_archive(
    base_name=str(ZIP_PATH.with_suffix("")),
    format="zip",
    root_dir=str(DRIVE_PATH.parent),
    base_dir=DRIVE_PATH.name,
)

zip_size_mb = ZIP_PATH.stat().st_size / (1024 ** 2)
zip_size_gb = zip_size_mb / 1024

print(f"\nZip created : {ZIP_PATH.name}")
print(f"Zip size    : {zip_size_mb:.1f} MB  ({zip_size_gb:.2f} GB)")

# ── Download ──────────────────────────────────────────────────
print("\n" + "=" * 70)
print("DOWNLOADING ZIP TO YOUR COMPUTER")
print("=" * 70)

print("Starting download...")
print("Check your browser's download bar.\n")

files.download(str(ZIP_PATH))

print("[PASS] Download started.")
print(f"\nFile : paza_whisper_kenyan_final.zip")
print(f"Size : {zip_size_mb:.1f} MB")